In [22]:
# THz Signal Processing — Hybrid Window S-Transform Pipeline

"""**Real-World THz Waveform Processing through the Full Pipeline:**

1. SigMF Data Loading (NIST 1.02 THz, 6 cm, BPSK, 5 GHz IF)
2. Adaptive S-Transform (p-optimization)
3. Parameter Optimization (gradient descent)
4. Garrote Denoising
5. Frame Comparison (Frame 1 vs Frame 2)
6. DOA Estimation via ROOT-MUSIC, TLS-ESPIRIT, DML, SPICE & MVDR

**Dataset:** NIST THz Modulated Data Collection  
- Carrier: 1.02 THz (LO = 1.0 THz, IF = 5 GHz)  
- Modulation: BPSK at 5 Gsym/s  
- ADC: 160 GS/s (ci16 format)  
- Link: 6 cm free-space path  
- Equipment: VDI MixAMC THz transceivers with conical horn antennas (26 dBi)
"""

'**Real-World THz Waveform Processing through the Full Pipeline:**\n\n1. SigMF Data Loading (NIST 1.02 THz, 6 cm, BPSK, 5 GHz IF)\n2. Adaptive S-Transform (p-optimization)\n3. Parameter Optimization (gradient descent)\n4. Garrote Denoising\n5. Frame Comparison (Frame 1 vs Frame 2)\n6. DOA Estimation via ROOT-MUSIC, TLS-ESPIRIT, DML, SPICE & MVDR\n\n**Dataset:** NIST THz Modulated Data Collection  \n- Carrier: 1.02 THz (LO = 1.0 THz, IF = 5 GHz)  \n- Modulation: BPSK at 5 Gsym/s  \n- ADC: 160 GS/s (ci16 format)  \n- Link: 6 cm free-space path  \n- Equipment: VDI MixAMC THz transceivers with conical horn antennas (26 dBi)\n'

In [23]:
!pip install scipy numpy -q
import json
import os
import re
import numpy as np
import warnings
import time as timer
from functools import lru_cache
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize_scalar, minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [24]:
## Cell 1: Concentration Measure

##Provides the Stankovic Concentration Measure (Rényi entropy-based) and Per-Frequency Concentration Measure for p-optimization.


In [25]:
"""
Concentration Measure Calculator
================================
Provides two concentration measure functions:

1. Stankovic Concentration Measure (Rényi entropy-based):
       M_p(S) = [ Σ |P(n,k)|^(1/p) ]^p
   where P(n,k) is the normalized energy distribution.
   Lower values → better energy concentration → sharper TFR.
   Used for overall distribution evaluation.

2. Per-Frequency Concentration Measure (Eq. 7):
       CM(f, p) = 1 / ∫ |S_x^p(t, f)|^q dt
   where p ∈ (0, 1] is the generalized S-Transform order (Eq. 5)
   and q ∈ (0, 0.25].
   Higher values → better concentration at that frequency.
   Used in the p_opt optimization algorithm (Eq. 8):
       p_opt(f) = argmax_p [CM(f, p)]

Import these from other scripts — no main execution here.
"""

import numpy as np


def concentration_measure(S_matrix, p):
    """
    Computes the Stankovic Concentration Measure of order p.

    Steps:
        1. Compute normalized energy distribution P(n,k)
        2. Apply Rényi-like entropy:  M_p = [ Σ (P + ε)^(1/p) ]^p

    Parameters
    ----------
    S_matrix : complex ndarray, shape (F, N)
        S-Transform output matrix.
    p : int or float
        Order of the concentration measure (default 4).
        Higher p → more sensitive to spread.

    Returns
    -------
    float
        Concentration measure value. Lower = better concentration.
    """
    magnitudes = np.abs(S_matrix)
    total_energy = np.sum(magnitudes**2)

    if total_energy == 0:
        return 0.0

    # Normalized energy distribution
    P = magnitudes**2 / total_energy

    epsilon = 1e-12  # prevent 0^(1/p) issues
    measure = (np.sum((P + epsilon) ** (1.0 / p))) ** p

    return float(measure)


def cm_per_frequency(S_matrix, q=0.25):
    """
    Per-frequency concentration measure from Equation 7:

        CM(f, p) = 1 / ∫ |S_x^p(t, f)|^q dt

    where p ∈ (0, 1] is the generalized S-Transform order (Eq. 5).
    The S-Transform S_x^p(t, f) must be pre-computed by the caller
    with the desired p and passed in as S_matrix.

    For each frequency row f:
        CM[f] = 1 / Σ_t |S[f, t]|^q

    Higher CM values → better energy concentration at that frequency.
    Used in the p_opt optimization algorithm (Eq. 8):
        p_opt(f) = argmax_p [CM(f, p)]

    Parameters
    ----------
    S_matrix : complex ndarray, shape (F, N)
        S-Transform output matrix, computed with a specific p.
    q : float
        Exponent parameter, must be in (0, 0.25].
        Smaller q → less sensitivity to amplitude differences.

    Returns
    -------
    cm : ndarray, shape (F,)
        Concentration measure value for each frequency bin.
        Higher = better concentration.

    Raises
    ------
    ValueError
        If q is not in the valid range (0, 0.25].
    """
    if not (0 < q <= 0.25):
        raise ValueError(f"q must be in (0, 0.25] — got {q}.")

    magnitudes = np.abs(S_matrix)           # |S_x^p(t, f)|
    integrand  = np.sum(magnitudes**q, axis=1)  # Σ_t |S[f,t]|^q per freq row

    # Avoid division by zero for silent frequency rows
    epsilon = 1e-30
    cm = 1.0 / (integrand + epsilon)

    return cm


In [26]:
## Cell 2: Hybrid Window S-Transform

"""FFT-based Generalized Hybrid Window S-Transform with Gaussian × Hyperbolic window product and adaptive per-frequency p-optimization."""


'FFT-based Generalized Hybrid Window S-Transform with Gaussian × Hyperbolic window product and adaptive per-frequency p-optimization.'

In [27]:
def validate_signal(signal):
    """Validates and sanitizes the input signal."""
    if signal is None:
        raise TypeError("Signal cannot be None.")
    signal = np.asarray(signal)
    if not np.iscomplexobj(signal):
        signal = signal.astype(np.float64)
    if signal.ndim != 1:
        raise ValueError(f"Signal must be 1-D — got {signal.ndim}-D array with shape {signal.shape}.")
    if len(signal) < 8:
        raise ValueError(f"Signal too short ({len(signal)} samples). Need at least 8 samples.")
    if np.sum(np.isnan(signal)) > 0:
        raise ValueError("Signal contains NaN values. Clean the data first.")
    if np.sum(np.isinf(signal)) > 0:
        raise ValueError("Signal contains Inf values. Clean the data first.")
    if np.allclose(signal, 0):
        warnings.warn("Signal is all zeros — the S-Transform output will be trivial.", UserWarning, stacklevel=2)
    return signal


def validate_parameters(gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq, p=1.0):
    """Validates all hybrid window parameters."""
    for name, val in {'gamma_GS': gamma_GS, 'gamma_HY_B': gamma_HY_B,
                       'gamma_HY_F': gamma_HY_F, 'lambda_sq': lambda_sq}.items():
        if not isinstance(val, (int, float)) or not np.isfinite(val) or val <= 0:
            raise ValueError(f"Parameter '{name}' must be a finite positive number — got {val}.")
    if not isinstance(p, (int, float)) or not np.isfinite(p) or p <= 0:
        raise ValueError(f"Parameter 'p' must be a finite positive number — got {p}.")


def hybrid_s_transform(signal, gamma_GS, gamma_HY_B, gamma_HY_F,
                       lambda_sq, p=1.0):
    """
    Computes the Generalized Hybrid Window S-Transform (FFT-based).
    Hybrid window = product of Gaussian and Hyperbolic windows.
    """
    signal = validate_signal(signal)
    validate_parameters(gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq, p)

    t0 = timer.perf_counter()
    N  = len(signal)
    H  = np.fft.fft(signal)
    H_concat = np.concatenate((H, H))

    S_hyb = np.zeros((N // 2, N), dtype=complex)
    t = np.fft.fftfreq(N) * N
    freq_bins = np.arange(N // 2)

    for f in range(N // 2):
        if f == 0:
            S_hyb[f, :] = np.mean(signal)
            continue

        # Generalized Gaussian window
        w_gs = (np.abs(f)**p / (np.sqrt(2*np.pi) * gamma_GS)) * \
               np.exp(-(f**(2*p) * t**2) / (2 * gamma_GS**2 * N**2))

        # Generalized Hyperbolic window with zeta translation
        zeta  = np.sqrt(((gamma_HY_B - gamma_HY_F)**2 * lambda_sq) /
                        (4 * gamma_HY_B * gamma_HY_F))
        term1 = ((gamma_HY_B + gamma_HY_F) / (2*gamma_HY_B*gamma_HY_F)) * (t - zeta)
        term2 = ((gamma_HY_B - gamma_HY_F) / (2*gamma_HY_B*gamma_HY_F)) * \
                np.sqrt((t - zeta)**2 + lambda_sq)
        X = term1 + term2
        w_hy = (2*np.abs(f)**p) / (np.sqrt(2*np.pi)*(gamma_HY_F + gamma_HY_B)) * \
               np.exp(-(f**(2*p) * X**2) / (2 * N**2))

        # Normalize
        gs_norm = np.sum(np.abs(w_gs))
        hy_norm = np.sum(np.abs(w_hy))
        if gs_norm < 1e-15 or hy_norm < 1e-15:
            continue
        w_gs /= gs_norm
        w_hy /= hy_norm

        # Hybrid = product
        w_hybrid = w_gs * w_hy
        hyb_norm = np.sum(np.abs(w_hybrid))
        if hyb_norm > 1e-15:
            w_hybrid /= hyb_norm

        shifted_H = H_concat[f:f + N]
        S_hyb[f, :] = np.fft.ifft(shifted_H * np.fft.fft(w_hybrid))

    return {
        'S_hybrid':     S_hyb,
        'frequencies':  freq_bins,
        'elapsed_sec':  timer.perf_counter() - t0,
    }


def time_resolution(S_matrix, freq_idx):
    """Half-maximum width of the energy envelope at a given frequency row."""
    envelope = np.abs(S_matrix[freq_idx, :])**2
    peak = np.max(envelope)
    if peak == 0:
        return len(envelope)
    indices = np.where(envelope >= peak / 2.0)[0]
    return 1 if len(indices) < 2 else int(indices[-1] - indices[0])


def detect_onset(S_matrix, freq_idx, threshold_factor=2.5, min_run=3, band_half=3):
    """Broadband threshold-based onset detector."""
    n_freqs, N = S_matrix.shape
    f_lo = max(1, freq_idx - band_half)
    f_hi = min(n_freqs - 1, freq_idx + band_half)
    envelope = np.sum(np.abs(S_matrix[f_lo:f_hi+1, :])**2, axis=0)
    baseline = envelope[:max(10, N // 5)]
    mean_b, std_b = np.mean(baseline), np.std(baseline)
    threshold = mean_b + (1e-6 if std_b < 1e-15 else threshold_factor * std_b)
    run = 0
    for i, val in enumerate(envelope):
        if val > threshold:
            run += 1
            if run >= min_run:
                return int(i - min_run + 1)
        else:
            run = 0
    return -1


def adaptive_s_transform(signal, gamma_GS, gamma_HY_B, gamma_HY_F,
                         lambda_sq, p_candidates=None, q=0.25):
    """
    Adaptive S-Transform via per-frequency p-optimization (Eqs. 7–9).
    """
    t0 = timer.perf_counter()
    signal = validate_signal(signal)
    N = len(signal)

    if p_candidates is None:
        p_candidates = np.linspace(0.1, 1.0, 10)
    p_candidates = np.asarray(p_candidates)

    n_freqs = N // 2
    CM_matrix = np.zeros((n_freqs, len(p_candidates)))
    S_all = []

    for j, p_val in enumerate(p_candidates):
        res_p = hybrid_s_transform(
            signal, gamma_GS=gamma_GS, gamma_HY_B=gamma_HY_B,
            gamma_HY_F=gamma_HY_F, lambda_sq=lambda_sq, p=float(p_val)
        )
        S_p = res_p['S_hybrid']
        S_all.append(S_p)
        CM_matrix[:, j] = cm_per_frequency(S_p, q=q)

    best_p_idx = np.argmax(CM_matrix, axis=1)
    p_opt_arr = p_candidates[best_p_idx]

    S_a = np.zeros((n_freqs, N), dtype=complex)
    for f_idx in range(n_freqs):
        S_a[f_idx, :] = S_all[best_p_idx[f_idx]][f_idx, :]

    return {
        'S_hybrid':     S_a,
        'p_opt':        {'Hybrid': p_opt_arr},
        'frequencies':  np.arange(n_freqs),
        'elapsed_sec':  timer.perf_counter() - t0,
    }



In [28]:
## Cell 3: SigMF Loader

"""Reads experimental THz datasets captured in the SigMF standard (.sigmf-data binary IQ samples + .sigmf-meta JSON metadata)."""


'Reads experimental THz datasets captured in the SigMF standard (.sigmf-data binary IQ samples + .sigmf-meta JSON metadata).'

In [29]:
"""
SigMF Loader — Read Signal Metadata Format (SigMF) Datasets
============================================================
Loads experimental THz datasets captured in the SigMF standard:

    .sigmf-data  — raw binary IQ samples (ci16 or cf32)
    .sigmf-meta  — JSON metadata (sample rate, data type, frequency, …)

This module provides:
    load_sigmf()       — load a single frame (data + metadata)
    extract_segment()  — extract a contiguous segment for S-Transform
    list_frames()      — discover all frame pairs in a dataset directory

Designed for the NIST THz Modulated Data collection (1–1.05 THz band).

References
----------
SigMF Specification: https://github.com/sigmf/SigMF/blob/main/sigmf-spec.md
NTIA Extensions:     https://github.com/NTIA/sigmf-ns-ntia
"""

import json
import os
import re
import numpy as np
import warnings


# ============================================================
# 1. LOAD A SINGLE SigMF FRAME
# ============================================================
def load_sigmf(data_path):
    """
    Loads a SigMF recording from a .sigmf-data file and its
    companion .sigmf-meta JSON metadata file.

    The data type is inferred from the metadata field
    ``core:datatype`` if present.  If absent (common in older
    NTIA datasets), the loader uses the ``core:sample_count``
    annotation to determine the correct interpretation:

        - ci16_le  →  complex int16, little-endian (4 bytes/sample)
        - cf32_le  →  complex float32, little-endian (8 bytes/sample)

    All samples are returned as complex128 (double-precision)
    for downstream numerical stability.

    Parameters
    ----------
    data_path : str
        Absolute or relative path to the ``.sigmf-data`` file.
        The ``.sigmf-meta`` file must exist alongside it with
        the same base name.

    Returns
    -------
    result : dict
        Keys:
        - ``signal``       : np.ndarray, complex128, shape (N,)
            The IQ samples as complex numbers.
        - ``sample_rate``  : float
            ADC sampling rate in Hz (from ``core:sample_rate``).
        - ``dt``           : float
            Sampling interval in picoseconds (= 1e12 / sample_rate).
        - ``center_freq``  : float
            Capture center / IF frequency in Hz.
        - ``rf_freq``      : float
            RF carrier frequency in Hz (LO + IF).
        - ``lo_freq``      : float
            Local oscillator frequency in Hz.
        - ``n_samples``    : int
            Number of complex IQ samples.
        - ``duration_ns``  : float
            Signal duration in nanoseconds.
        - ``description``  : str
            Human-readable description from metadata.
        - ``datatype``     : str
            Detected data type string (e.g. 'ci16_le').
        - ``metadata``     : dict
            Full parsed JSON metadata.

    Raises
    ------
    FileNotFoundError
        If the .sigmf-data or .sigmf-meta file does not exist.
    ValueError
        If the data type cannot be determined or data is corrupt.
    """
    # ── Resolve paths ──
    data_path = os.path.abspath(data_path)
    if not os.path.isfile(data_path):
        raise FileNotFoundError(f"SigMF data file not found: {data_path}")

    meta_path = data_path.replace('.sigmf-data', '.sigmf-meta')
    if not os.path.isfile(meta_path):
        raise FileNotFoundError(
            f"SigMF metadata file not found: {meta_path}\n"
            f"Expected alongside: {data_path}"
        )

    # ── Read metadata ──
    with open(meta_path, 'r') as f:
        metadata = json.load(f)

    global_meta = metadata.get('global', {})
    captures    = metadata.get('captures', [{}])
    annotations = metadata.get('annotations', [])

    # ── Extract key parameters ──
    sample_rate = global_meta.get('core:sample_rate', None)
    if sample_rate is None:
        raise ValueError("Metadata missing 'core:sample_rate'.")

    description = global_meta.get('core:description', 'No description')

    # Center / IF frequency
    center_freq = captures[0].get('core:frequency', 0.0) if captures else 0.0

    # LO frequency (from sensor metadata)
    sensor = global_meta.get('ntia-sensor:sensor', {})
    lo_freq = sensor.get('local_oscillator_frequency', 0.0)

    # RF = LO + IF
    rf_freq = lo_freq + center_freq if lo_freq > 0 else center_freq

    # ── Determine data type ──
    datatype = global_meta.get('core:datatype', None)

    # Get annotated sample count (if available)
    annotated_count = None
    for ann in annotations:
        if 'core:sample_count' in ann:
            annotated_count = ann['core:sample_count']
            break

    file_size = os.path.getsize(data_path)

    if datatype is not None:
        # Explicit data type in metadata
        datatype = datatype.lower().strip()
    else:
        # Infer from file size and annotated sample count
        datatype = _infer_datatype(file_size, annotated_count)

    # ── Read binary data ──
    signal = _read_binary(data_path, datatype, file_size)

    # ── Validate against annotated count ──
    if annotated_count is not None and len(signal) != annotated_count:
        warnings.warn(
            f"Sample count mismatch: loaded {len(signal)} samples "
            f"but metadata says {annotated_count}. "
            f"Data type detection may be wrong.",
            UserWarning, stacklevel=2
        )

    # ── Compute derived quantities ──
    n_samples   = len(signal)
    dt_ps       = 1e12 / sample_rate          # sampling interval in ps
    duration_ns = n_samples / sample_rate * 1e9

    # ── Extract emitter waveform info (if present) ──
    emitter_info = {}
    emitters = global_meta.get('ntia-emitter:emitters', [])
    if emitters:
        e = emitters[0]
        wf = e.get('waveform', {})
        emitter_info = {
            'modulation':    wf.get('modulation_type', 'unknown'),
            'symbol_rate':   wf.get('symbol_rate', 0.0),
            'n_pilot_bits':  wf.get('number_of_pilot_bits', 0),
            'amplitude_mV':  e.get('amplitude', 0.0),
            'tx_sample_rate': e.get('sample_rate', 0.0),
            'tx_lo_freq':    e.get('local_oscillator_frequency', 0.0),
            'if_freq':       e.get('intermediate_frequency', 0.0),
        }

    # ── Extract filter info (if present) ──
    filter_info = {}
    for ann in annotations:
        if ann.get('ntia-core:annotation_type') == 'DigitalFilterAnnotation':
            filter_info = {
                'filter_type': ann.get('ntia-algorithm:filter_type', ''),
                'sps':         ann.get('sps', 0),
                'window_type': ann.get('window_type', ''),
                'beta':        ann.get('beta', 0.0),
                'span':        ann.get('span', 0),
            }
            break

    # ── Link distance ──
    link_distance = captures[0].get('link_distance', None) if captures else None

    return {
        'signal':        signal,
        'sample_rate':   sample_rate,
        'dt':            dt_ps,
        'center_freq':   center_freq,
        'rf_freq':       rf_freq,
        'lo_freq':       lo_freq,
        'n_samples':     n_samples,
        'duration_ns':   duration_ns,
        'description':   description,
        'datatype':      datatype,
        'metadata':      metadata,
        'emitter_info':  emitter_info,
        'filter_info':   filter_info,
        'link_distance': link_distance,
    }


# ============================================================
# 2. EXTRACT A SEGMENT FOR S-TRANSFORM
# ============================================================
def extract_segment(signal, n_samples=1024, center='peak_energy',
                    component='real'):
    """
    Extracts a contiguous segment from a complex IQ signal,
    suitable for the S-Transform pipeline.

    The S-Transform creates an (N//2 × N) matrix, so large N
    becomes impractical.  This function selects a meaningful
    segment centred on the strongest part of the signal.

    Parameters
    ----------
    signal : np.ndarray, complex, shape (L,)
        Full IQ signal from load_sigmf().
    n_samples : int
        Length of the segment to extract (default 512).
        Should be a power of 2 for FFT efficiency.
    center : str or int
        How to choose the segment centre:
        - 'peak_energy' : centre on the sample with maximum
          instantaneous power |x[n]|² (default).
        - 'middle' : centre of the signal.
        - int : explicit sample index.
    component : str
        Which part of the complex signal to return:
        - 'real'      : np.real(segment)  — recommended for BPSK
        - 'imag'      : np.imag(segment)
        - 'magnitude' : np.abs(segment)
        - 'complex'   : segment unchanged (complex-valued)

    Returns
    -------
    result : dict
        Keys:
        - ``segment``     : np.ndarray, shape (n_samples,)
            The extracted segment (type depends on `component`).
        - ``segment_iq``  : np.ndarray, complex, shape (n_samples,)
            The raw complex IQ segment (always complex).
        - ``start_idx``   : int
            Start index in the original signal.
        - ``end_idx``     : int
            End index (exclusive) in the original signal.
        - ``center_idx``  : int
            Centre index in the original signal.
        - ``component``   : str
            Which component was extracted.

    Raises
    ------
    ValueError
        If n_samples exceeds the signal length.
    """
    L = len(signal)

    if n_samples > L:
        warnings.warn(
            f"Requested segment ({n_samples}) exceeds signal length ({L}). "
            f"Using full signal.",
            UserWarning, stacklevel=2
        )
        n_samples = L

    # ── Determine centre ──
    if isinstance(center, int):
        center_idx = center
    elif center == 'peak_energy':
        # Smooth power envelope to avoid picking a noise spike
        power = np.abs(signal) ** 2
        # Use a running average over ~1 symbol period if signal is long enough
        kernel_size = min(32, L // 4)
        if kernel_size > 1:
            kernel = np.ones(kernel_size) / kernel_size
            power_smooth = np.convolve(power, kernel, mode='same')
        else:
            power_smooth = power
        center_idx = int(np.argmax(power_smooth))
    elif center == 'middle':
        center_idx = L // 2
    else:
        raise ValueError(
            f"center must be 'peak_energy', 'middle', or an int — got {center!r}"
        )

    # ── Compute start/end with boundary clamping ──
    half = n_samples // 2
    start = center_idx - half
    end   = start + n_samples

    # Clamp to signal boundaries
    if start < 0:
        start = 0
        end   = n_samples
    if end > L:
        end   = L
        start = L - n_samples

    segment_iq = signal[start:end].copy()

    # ── Extract requested component ──
    if component == 'real':
        segment = np.real(segment_iq).astype(np.float64)
    elif component == 'imag':
        segment = np.imag(segment_iq).astype(np.float64)
    elif component == 'magnitude':
        segment = np.abs(segment_iq).astype(np.float64)
    elif component == 'complex':
        segment = segment_iq
    else:
        raise ValueError(
            f"component must be 'real', 'imag', 'magnitude', or 'complex' "
            f"— got {component!r}"
        )

    return {
        'segment':     segment,
        'segment_iq':  segment_iq,
        'start_idx':   start,
        'end_idx':     end,
        'center_idx':  center_idx,
        'component':   component,
    }


# ============================================================
# 3. LIST ALL FRAMES IN A DATASET DIRECTORY
# ============================================================
def list_frames(dataset_dir):
    """
    Discovers all SigMF frame pairs (.sigmf-data + .sigmf-meta)
    in a dataset directory and returns them sorted by frame number.

    Parameters
    ----------
    dataset_dir : str
        Path to the dataset directory (e.g.,
        '1-1.05 THz/1.02THz_6cm_2PSK_IF_5G_5GSyms_600mV/').

    Returns
    -------
    frames : list of dict
        Each dict has keys:
        - ``data_path`` : str — absolute path to .sigmf-data
        - ``meta_path`` : str — absolute path to .sigmf-meta
        - ``frame_num`` : int — extracted frame number
        - ``basename``  : str — file basename without extension
    """
    dataset_dir = os.path.abspath(dataset_dir)
    if not os.path.isdir(dataset_dir):
        raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

    frames = []
    for fname in sorted(os.listdir(dataset_dir)):
        if fname.endswith('.sigmf-data'):
            data_path = os.path.join(dataset_dir, fname)
            meta_path = data_path.replace('.sigmf-data', '.sigmf-meta')

            if not os.path.isfile(meta_path):
                warnings.warn(
                    f"Missing .sigmf-meta for {fname} — skipping.",
                    UserWarning, stacklevel=2
                )
                continue

            basename = fname.replace('.sigmf-data', '')

            # Extract frame number from filename
            match = re.search(r'_(\d+)$', basename)
            frame_num = int(match.group(1)) if match else 0

            frames.append({
                'data_path': data_path,
                'meta_path': meta_path,
                'frame_num': frame_num,
                'basename':  basename,
            })

    # Sort by frame number
    frames.sort(key=lambda f: f['frame_num'])

    return frames


# ============================================================
# INTERNAL HELPERS
# ============================================================
def _infer_datatype(file_size, annotated_count):
    """
    Infers the SigMF data type from file size and annotated
    sample count when core:datatype is not specified.

    Logic:
        bytes_per_sample = file_size / annotated_count
        4 bytes → ci16_le  (2 bytes I + 2 bytes Q)
        8 bytes → cf32_le  (4 bytes I + 4 bytes Q)
    """
    if annotated_count is not None and annotated_count > 0:
        bps = file_size / annotated_count
        if abs(bps - 4.0) < 0.01:
            return 'ci16_le'
        elif abs(bps - 8.0) < 0.01:
            return 'cf32_le'
        else:
            raise ValueError(
                f"Cannot infer data type: file_size={file_size}, "
                f"annotated_count={annotated_count}, "
                f"bytes_per_sample={bps:.2f} (expected 4 or 8)."
            )
    else:
        # No annotated count — try cf32 first (more common in modern recordings)
        if file_size % 8 == 0:
            warnings.warn(
                "No core:datatype or core:sample_count in metadata. "
                "Assuming cf32_le (complex float32).",
                UserWarning, stacklevel=2
            )
            return 'cf32_le'
        elif file_size % 4 == 0:
            warnings.warn(
                "No core:datatype or core:sample_count in metadata. "
                "Assuming ci16_le (complex int16).",
                UserWarning, stacklevel=2
            )
            return 'ci16_le'
        else:
            raise ValueError(
                f"Cannot infer data type: file_size={file_size} "
                f"is not divisible by 4 or 8."
            )


def _read_binary(data_path, datatype, file_size):
    """
    Reads raw binary IQ samples and converts to complex128.

    Supported types:
        ci16_le  — interleaved int16 little-endian (I, Q, I, Q, …)
        cf32_le  — interleaved float32 little-endian
        ri16_le  — real int16 (no imaginary part)
        rf32_le  — real float32 (no imaginary part)
    """
    datatype = datatype.lower().strip()

    if datatype in ('ci16_le', 'ci16'):
        # Read as interleaved int16 pairs
        raw = np.fromfile(data_path, dtype=np.dtype('<i2'))
        if len(raw) % 2 != 0:
            raise ValueError(
                f"ci16 data has odd number of int16 values ({len(raw)}). "
                f"Expected even (I,Q pairs)."
            )
        # Reshape to (N, 2) then combine as complex
        iq = raw.reshape(-1, 2).astype(np.float64)
        signal = iq[:, 0] + 1j * iq[:, 1]

        # Normalize int16 range [-32768, 32767] to [-1, 1]
        signal = signal / 32768.0

    elif datatype in ('cf32_le', 'cf32'):
        signal = np.fromfile(data_path, dtype=np.complex64)
        signal = signal.astype(np.complex128)

    elif datatype in ('ri16_le', 'ri16'):
        raw = np.fromfile(data_path, dtype=np.dtype('<i2'))
        signal = raw.astype(np.float64) / 32768.0
        signal = signal + 0j  # make complex with zero imaginary

    elif datatype in ('rf32_le', 'rf32'):
        raw = np.fromfile(data_path, dtype=np.dtype('<f4'))
        signal = raw.astype(np.float64)
        signal = signal + 0j  # make complex with zero imaginary

    else:
        raise ValueError(
            f"Unsupported SigMF datatype: '{datatype}'. "
            f"Supported: ci16_le, cf32_le, ri16_le, rf32_le."
        )

    return signal


# ============================================================
# MAIN — Smoke Test


In [30]:
## Cell 4: Denoising

"""Non-Negative Garrote shrinkage denoising with MAD noise estimation, minimax-optimal threshold (Λ*_n), and risk functions."""


'Non-Negative Garrote shrinkage denoising with MAD noise estimation, minimax-optimal threshold (Λ*_n), and risk functions.'

In [31]:
import numpy as np
from functools import lru_cache
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize_scalar, minimize


# ============================================================
# STEP 1: EXTRACT THE MAGNITUDE
# ============================================================
def extract_magnitude(S_opt):
    """
    Extracts the real-valued magnitude matrix from a complex-valued
    S-Transform time-frequency representation.

        A(t, f) = |S_opt(t, f)|

    Noise statistics require real-valued inputs — this function converts
    the complex S-Transform output into its element-wise absolute value.

    Parameters
    ----------
    S_opt : np.ndarray, complex, shape (n_freqs, n_times)
        Complex-valued optimized S-Transform matrix (e.g. from
        adaptive_s_transform or hybrid_s_transform).  Rows index
        frequency bins and columns index time samples.

    Returns
    -------
    A : np.ndarray, float64, shape (n_freqs, n_times)
        Real-valued magnitude matrix  A(t, f) = |S_opt(t, f)|.

    Raises
    ------
    TypeError
        If S_opt is not a numpy array.
    ValueError
        If S_opt is not a 2-D matrix.
    """
    if not isinstance(S_opt, np.ndarray):
        raise TypeError(
            f"S_opt must be a numpy ndarray — got {type(S_opt).__name__}."
        )
    if S_opt.ndim != 2:
        raise ValueError(
            f"S_opt must be a 2-D matrix — got {S_opt.ndim}-D array with shape {S_opt.shape}."
        )

    A = np.abs(S_opt)
    return A


# ============================================================
# STEP 2: ESTIMATE NOISE VARIANCE (σ) VIA MAD
# ============================================================
def estimate_noise_variance(A):
    """
    Estimates the noise standard deviation σ from the magnitude matrix
    using the Median Absolute Deviation (MAD).

        σ = median(|A − median(A)|) / 0.6745

    The entire 2-D magnitude matrix is treated as a single flat list of
    values for this statistical calculation.  The 0.6745 scale factor is
    the 75th percentile of the standard normal distribution, which makes
    the MAD a consistent estimator of σ for Gaussian noise.

    Parameters
    ----------
    A : np.ndarray, float, shape (n_freqs, n_times)
        Real-valued magnitude matrix (output of extract_magnitude).

    Returns
    -------
    sigma : float
        Estimated noise standard deviation.

    Raises
    ------
    TypeError
        If A is not a numpy array.
    ValueError
        If A is not a 2-D matrix.
    """
    if not isinstance(A, np.ndarray):
        raise TypeError(
            f"A must be a numpy ndarray — got {type(A).__name__}."
        )
    if A.ndim != 2:
        raise ValueError(
            f"A must be a 2-D matrix — got {A.ndim}-D array with shape {A.shape}."
        )

    # Flatten the 2-D matrix into a single 1-D vector
    flat = A.ravel()

    # Median of all magnitude values
    med = np.median(flat)

    # Absolute deviations from the median
    abs_dev = np.abs(flat - med)

    # MAD = median of absolute deviations
    mad = np.median(abs_dev)

    # Scale to get a consistent estimator of σ for Gaussian noise
    sigma = mad / 0.6745

    return sigma


# ============================================================
# STEP 3: UNIVERSAL THRESHOLD (λ)
# ============================================================
def universal_threshold(sigma, A):
    """
    Computes the universal (VisuShrink) threshold for denoising.

        λ = σ · √(2 ln N)

    where N = total number of elements in the 2-D magnitude matrix
    (N = n_freqs × n_times).

    Parameters
    ----------
    sigma : float
        Estimated noise standard deviation (output of estimate_noise_variance).
    A : np.ndarray, float, shape (n_freqs, n_times)
        Real-valued magnitude matrix (used only to determine N).

    Returns
    -------
    lam : float
        Universal threshold λ.

    Raises
    ------
    TypeError
        If sigma is not numeric or A is not a numpy array.
    ValueError
        If sigma is negative or A is not 2-D.
    """
    if not isinstance(sigma, (int, float)):
        raise TypeError(
            f"sigma must be numeric — got {type(sigma).__name__}."
        )
    if sigma < 0:
        raise ValueError(f"sigma must be ≥ 0 — got {sigma}.")
    if not isinstance(A, np.ndarray):
        raise TypeError(
            f"A must be a numpy ndarray — got {type(A).__name__}."
        )
    if A.ndim != 2:
        raise ValueError(
            f"A must be a 2-D matrix — got {A.ndim}-D array with shape {A.shape}."
        )

    # Total number of pixels in the TFR
    N = A.shape[1]  # n_times (independent degrees of freedom)
    
    # Universal threshold: λ = σ √(2 ln N)
    lam = sigma * np.sqrt(2 * np.log(N))

    return lam


# ============================================================
# STEP 4: NON-NEGATIVE GARROTE THRESHOLDING  (Equation 1)
# ============================================================
def garrote_threshold(A, lam):
    """
    Applies the Non-Negative Garrote (NNG) shrinkage function to
    the magnitude matrix A(t, f).

        S_G(x) = x · {1 − (λ/x)²}₊

    which expands piece-wise as:

        S_G(x) = 0              if |x| ≤ λ
        S_G(x) = x − λ²/x      if |x| > λ

    Unlike hard thresholding (binary 0/1 mask), the garrote produces
    a continuous shrinkage: coefficients just above λ are attenuated
    smoothly, reducing Gibbs-like artefacts while still zeroing out
    noise-dominated pixels.

    Parameters
    ----------
    A : np.ndarray, float, shape (n_freqs, n_times)
        Real-valued magnitude matrix (output of extract_magnitude).
    lam : float
        Threshold λ (output of universal_threshold or actual_threshold).

    Returns
    -------
    A_garrote : np.ndarray, float64, shape (n_freqs, n_times)
        Garrote-thresholded magnitude matrix.
        — Pixels with |A| ≤ λ  →  0
        — Pixels with |A| > λ  →  A − λ²/A

    Raises
    ------
    TypeError
        If A is not a numpy array or lam is not numeric.
    ValueError
        If A is not 2-D or lam is negative.
    """
    if not isinstance(A, np.ndarray):
        raise TypeError(
            f"A must be a numpy ndarray — got {type(A).__name__}."
        )
    if A.ndim != 2:
        raise ValueError(
            f"A must be a 2-D matrix — got {A.ndim}-D array with shape {A.shape}."
        )
    if not isinstance(lam, (int, float)):
        raise TypeError(
            f"lam must be numeric — got {type(lam).__name__}."
        )
    if lam < 0:
        raise ValueError(f"lam must be ≥ 0 — got {lam}.")

    # -----------------------------------------------------------
    # Garrote shrinkage:  S_G(x) = x · max(1 − (λ/x)², 0)
    #
    #   • Where |A| > λ  →  A − λ²/A   (smooth shrinkage)
    #   • Where |A| ≤ λ  →  0           (noise removal)
    # -----------------------------------------------------------
    A_garrote = np.where(
        A > lam,
        A - (lam ** 2) / A,   # signal region: shrink smoothly
        0.0                    # noise region:  zero out
    )

    return A_garrote


# ============================================================
# STEP 4a: GARROTE SHRINKAGE RATIO  G(t, f)
# ============================================================
def garrote_shrinkage_ratio(A, lam_actual):
    """
    Computes the garrote shrinkage ratio G(t, f) using the optimal
    data-scaled threshold λ_actual.

        G(t, f) = max(1 − (λ_actual / A(t,f))², 0)

    Values of G lie in [0, 1):
      • G = 0    where  A ≤ λ_actual    (noise removed)
      • G → 1    where  A ≫ λ_actual    (strong signal untouched)

    Parameters
    ----------
    A : np.ndarray, float, shape (n_freqs, n_times)
        Real-valued magnitude matrix (output of extract_magnitude).
    lam_actual : float
        Data-scaled optimal threshold  σ · Λ*_n  (output of
        actual_threshold).

    Returns
    -------
    G : np.ndarray, float64, shape (n_freqs, n_times)
        Shrinkage ratio matrix (values in [0, 1)).

    Raises
    ------
    TypeError
        If A is not a numpy array or lam_actual is not numeric.
    ValueError
        If A is not 2-D or lam_actual is negative.
    """
    if not isinstance(A, np.ndarray):
        raise TypeError(
            f"A must be a numpy ndarray — got {type(A).__name__}."
        )
    if A.ndim != 2:
        raise ValueError(
            f"A must be a 2-D matrix — got {A.ndim}-D array with shape {A.shape}."
        )
    if not isinstance(lam_actual, (int, float)):
        raise TypeError(
            f"lam_actual must be numeric — got {type(lam_actual).__name__}."
        )
    if lam_actual < 0:
        raise ValueError(f"lam_actual must be ≥ 0 — got {lam_actual}.")

    G = np.where(
        A > lam_actual,
        1.0 - (lam_actual / A) ** 2,   # shrinkage ratio
        0.0                             # zero out noise
    )

    return G


# ============================================================
# STEP 4b: DENOISE COMPLEX MATRIX VIA GARROTE
# ============================================================
def garrote_denoise(S_opt, G):
    """
    Denoises the complex S-Transform by element-wise multiplication
    with the garrote shrinkage ratio G.

        S_clean(t, f) = S_opt(t, f) · G(t, f)

    Phase is fully preserved.  Magnitudes are smoothly shrunk by G,
    which ranges from 0 (noise removed) to nearly 1 (strong signal
    untouched).

    Parameters
    ----------
    S_opt : np.ndarray, complex, shape (n_freqs, n_times)
        Original complex-valued S-Transform matrix.
    G : np.ndarray, float, shape (n_freqs, n_times)
        Garrote shrinkage ratio (output of garrote_shrinkage_ratio).

    Returns
    -------
    S_clean : np.ndarray, complex, shape (n_freqs, n_times)
        Denoised complex S-Transform.

    Raises
    ------
    TypeError
        If S_opt or G is not a numpy array.
    ValueError
        If shapes do not match or arrays are not 2-D.
    """
    if not isinstance(S_opt, np.ndarray):
        raise TypeError(
            f"S_opt must be a numpy ndarray — got {type(S_opt).__name__}."
        )
    if not isinstance(G, np.ndarray):
        raise TypeError(
            f"G must be a numpy ndarray — got {type(G).__name__}."
        )
    if S_opt.ndim != 2:
        raise ValueError(
            f"S_opt must be 2-D — got {S_opt.ndim}-D with shape {S_opt.shape}."
        )
    if G.ndim != 2:
        raise ValueError(
            f"G must be 2-D — got {G.ndim}-D with shape {G.shape}."
        )
    if S_opt.shape != G.shape:
        raise ValueError(
            f"Shape mismatch: S_opt {S_opt.shape} vs G {G.shape}."
        )

    S_clean = S_opt * G

    return S_clean



# ============================================================
# STEP 5: RISK HELPER  A_λ(θ)   (Equation 2.2)
# ============================================================
def A_lambda(lam, theta):
    """
    Computes the risk auxiliary function A_λ(θ) from Proposition 1.

                     ∞
        A_λ(θ)  =  ∫   { [φ(x − θ) − φ(x + θ)] / x }  dx
                     λ

    where φ(·) is the standard Gaussian probability density function.

    Parameters
    ----------
    lam : float
        Threshold λ  (lower limit of integration, ≥ 0).
    theta : float
        Signal-to-noise parameter θ.

    Returns
    -------
    value : float
        Numerical value of A_λ(θ).
    """
    if not isinstance(lam, (int, float)):
        raise TypeError(
            f"lam must be numeric — got {type(lam).__name__}."
        )
    if not isinstance(theta, (int, float)):
        raise TypeError(
            f"theta must be numeric — got {type(theta).__name__}."
        )
    if lam < 0:
        raise ValueError(f"lam must be ≥ 0 — got {lam}.")

    # Integrand:  [φ(x − θ) − φ(x + θ)] / x
    def integrand(x):
        return (norm.pdf(x - theta) - norm.pdf(x + theta)) / x

    value, _ = quad(integrand, lam, np.inf)
    return value


# ============================================================
# STEP 6: RISK HELPER  B_λ(θ)   (Equation 2.3)
# ============================================================
def B_lambda(lam, theta):
    """
    Computes the risk auxiliary function B_λ(θ) from Proposition 1.

                     ∞
        B_λ(θ)  =  ∫   { [φ(x − θ) + φ(x + θ)] / x² }  dx
                     λ

    where φ(·) is the standard Gaussian probability density function.

    Parameters
    ----------
    lam : float
        Threshold λ  (lower limit of integration, ≥ 0).
    theta : float
        Signal-to-noise parameter θ.

    Returns
    -------
    value : float
        Numerical value of B_λ(θ).
    """
    if not isinstance(lam, (int, float)):
        raise TypeError(
            f"lam must be numeric — got {type(lam).__name__}."
        )
    if not isinstance(theta, (int, float)):
        raise TypeError(
            f"theta must be numeric — got {type(theta).__name__}."
        )
    if lam < 0:
        raise ValueError(f"lam must be ≥ 0 — got {lam}.")

    # Integrand:  [φ(x − θ) + φ(x + θ)] / x²
    def integrand(x):
        return (norm.pdf(x - theta) + norm.pdf(x + theta)) / (x ** 2)

    value, _ = quad(integrand, lam, np.inf)
    return value



# ============================================================
# STEP 8: HARD-THRESHOLD RISK  R_λ^H(θ)
# ============================================================
def R_lambda_H(lam, theta):
    """
    Risk (MSE) of the hard-threshold estimator.

        R_λ^H(θ) = 1 + (θ² − 1){Φ(λ−θ) − Φ(−λ−θ)}
                   + (λ+θ) φ(λ+θ) + (λ−θ) φ(λ−θ)

    Parameters
    ----------
    lam : float
        Threshold λ (≥ 0).
    theta : float
        Signal-to-noise parameter θ.

    Returns
    -------
    value : float
        R_λ^H(θ).
    """
    phi_lm = norm.pdf(lam - theta)       # φ(λ − θ)
    phi_lp = norm.pdf(lam + theta)       # φ(λ + θ)
    Phi_lm = norm.cdf(lam - theta)       # Φ(λ − θ)
    Phi_neg_lm = norm.cdf(-lam - theta)  # Φ(−λ − θ)

    value = (1
             + (theta ** 2 - 1) * (Phi_lm - Phi_neg_lm)
             + (lam + theta) * phi_lp
             + (lam - theta) * phi_lm)
    return value


# ============================================================
# STEP 9: GARROTE RISK  R_λ^G(θ)   (Equation 2.4)
# ============================================================
def R_lambda_G(lam, theta):
    """
    Risk (MSE) of the Non-Negative Garrote estimator (Eq. 2.4).

        R_λ^G(θ) = R_λ^H(θ) + 2θ λ² A_λ(θ) + λ⁴ B_λ(θ)
                   − 2λ² {1 − Φ(λ−θ) + Φ(−λ−θ)}

    This combines:
      • R_λ^H  — hard-threshold risk  (Step 8)
      • A_λ    — risk integral helper  (Step 5, Eq. 2.2)
      • B_λ    — risk integral helper  (Step 6, Eq. 2.3)
      • Φ      — standard Gaussian CDF

    Parameters
    ----------
    lam : float
        Threshold λ (≥ 0).
    theta : float
        Signal-to-noise parameter θ.

    Returns
    -------
    risk : float
        R_λ^G(θ) — mean-squared-error risk of the garrote shrinkage
        rule at signal level θ.
    """
    # Hard-threshold risk
    R_H = R_lambda_H(lam, theta)

    # Risk integral helpers (Eqs. 2.2 & 2.3)
    A = A_lambda(lam, theta)
    B = B_lambda(lam, theta)

    # Gaussian CDF terms
    Phi_lm = norm.cdf(lam - theta)       # Φ(λ − θ)
    Phi_neg_lm = norm.cdf(-lam - theta)  # Φ(−λ − θ)

    # Equation 2.4
    risk = (R_H
            + 2 * theta * lam ** 2 * A
            + lam ** 4 * B
            - 2 * lam ** 2 * (1 - Phi_lm + Phi_neg_lm))

    return risk


# ============================================================
# STEP 9: OPTIMAL THRESHOLD  Λ*_n   (Equation 3.3)
# ============================================================
@lru_cache(maxsize=32)
def optimal_lambda(n, theta_max=10.0, lam_bounds=(0.01, 10.0)):
    """
    Finds the minimax-optimal threshold Λ*_n (Eq. 3.3) for the
    Non-Negative Garrote shrinkage rule.

                          ⎧  R_λ^G(θ)           ⎫
        Λ*_n  ≡  inf  sup ⎨ ─────────────────── ⎬
                   λ   θ  ⎩ n⁻¹ + (θ² ∧ 1)     ⎭

    where  θ² ∧ 1 = min(θ², 1).

    The inner supremum finds the worst-case θ (the signal level
    that makes the normalised risk largest).  The outer infimum
    picks the λ that minimises that worst case.

    Parameters
    ----------
    n : int
        Number of observations (sample size).  Determines the
        bias term n⁻¹ in the denominator.
    theta_max : float, optional
        Upper bound of the θ search range for the inner sup.
        Default is 10.0.
    lam_bounds : tuple of float, optional
        (lower, upper) search bounds for λ.
        Default is (0.01, 10.0).

    Returns
    -------
    lam_star : float
        Optimal threshold Λ*_n.
    min_max_risk : float
        The minimised worst-case normalised risk value.
    """
    if not isinstance(n, (int, float)) or n <= 0:
        raise ValueError(f"n must be a positive number — got {n}.")

    inv_n = 1.0 / n

    # ----------------------------------------------------------
    # Normalised risk:  R_λ^G(θ) / [ n⁻¹ + min(θ², 1) ]
    # ----------------------------------------------------------
    def normalised_risk(theta, lam):
        risk = R_lambda_G(lam, theta)
        denom = inv_n + min(theta ** 2, 1.0)
        return risk / denom

    # ----------------------------------------------------------
    # Inner problem:  sup_θ  normalised_risk(θ, λ)
    # Solved by minimising the negative over a grid + local search.
    # ----------------------------------------------------------
    def worst_case_risk(lam):
        # Coarse grid to avoid local optima
        thetas = np.linspace(0.0, theta_max, 200)
        grid_vals = [normalised_risk(th, lam) for th in thetas]
        best_idx = int(np.argmax(grid_vals))

        # Refine with local optimisation (negate for minimiser)
        result = minimize_scalar(
            lambda th: -normalised_risk(th, lam),
            bounds=(max(0.0, thetas[best_idx] - 0.5),
                    min(theta_max, thetas[best_idx] + 0.5)),
            method='bounded'
        )
        return -result.fun  # sup value

    # ----------------------------------------------------------
    # Outer problem:  inf_λ  worst_case_risk(λ)
    # ----------------------------------------------------------
    result = minimize_scalar(
        worst_case_risk,
        bounds=lam_bounds,
        method='bounded'
    )

    lam_star = result.x
    min_max_risk = result.fun

    return lam_star, min_max_risk


# ============================================================
# STEP 10: ACTUAL THRESHOLD  λ_actual = σ · Λ*_n
# ============================================================
def actual_threshold(sigma, n):
    """
    Computes the data-scaled optimal threshold for garrote shrinkage.

        λ_actual  =  σ · Λ*_n

    The minimax-optimal Λ*_n (Eq. 3.3) is derived under the assumption
    that σ = 1.  Multiplying by the estimated noise standard deviation σ
    (from Step 2) rescales it to the actual noise level of the data.

    Parameters
    ----------
    sigma : float
        Estimated noise standard deviation (output of estimate_noise_variance).
    n : int
        Number of observations N = n_freqs × n_times  (= A.size).

    Returns
    -------
    lam_actual : float
        Data-scaled optimal threshold  σ · Λ*_n.
    lam_star : float
        The normalised (σ = 1) minimax threshold Λ*_n.
    min_max_risk : float
        The minimised worst-case normalised risk.

    Raises
    ------
    TypeError
        If sigma is not numeric.
    ValueError
        If sigma is negative or n ≤ 0.
    """
    if not isinstance(sigma, (int, float)):
        raise TypeError(
            f"sigma must be numeric — got {type(sigma).__name__}."
        )
    if sigma < 0:
        raise ValueError(f"sigma must be ≥ 0 — got {sigma}.")
    if not isinstance(n, (int, float)) or n <= 0:
        raise ValueError(f"n must be a positive number — got {n}.")

    # Minimax-optimal threshold in the σ = 1 domain
    lam_star, min_max_risk = optimal_lambda(n)

    # Scale to the actual noise level
    lam_actual = sigma * lam_star

    return lam_actual, lam_star, min_max_risk


# ============================================================
# STEP 11: DENOISE THE COMPLEX MATRIX  (Garrote Shrinkage)
# ============================================================
def apply_mask(S_opt, lam_actual):
    """
    Denoises the complex S-Transform using Non-Negative Garrote
    shrinkage with the optimal data-scaled threshold λ_actual.

    Internally computes:
      1. A(t, f)  =  |S_opt(t, f)|                — magnitude
      2. G(t, f)  =  max(1 − (λ_actual / A)², 0)  — garrote shrinkage ratio
      3. S_clean   =  S_opt · G                    — denoised output

    Phase is fully preserved.  Coefficients with |S_opt| ≤ λ_actual
    are zeroed (noise removed);  those with |S_opt| > λ_actual are
    smoothly attenuated.

    Parameters
    ----------
    S_opt : np.ndarray, complex, shape (n_freqs, n_times)
        Original complex-valued S-Transform matrix.
    lam_actual : float
        Data-scaled optimal threshold  σ · Λ*_n  (output of
        actual_threshold).

    Returns
    -------
    S_clean : np.ndarray, complex, shape (n_freqs, n_times)
        Denoised complex S-Transform.
    G : np.ndarray, float64, shape (n_freqs, n_times)
        Garrote shrinkage ratio matrix (values in [0, 1)).

    Raises
    ------
    TypeError
        If S_opt is not a numpy array or lam_actual is not numeric.
    ValueError
        If S_opt is not 2-D or lam_actual is negative.
    """
    if not isinstance(S_opt, np.ndarray):
        raise TypeError(
            f"S_opt must be a numpy ndarray — got {type(S_opt).__name__}."
        )
    if S_opt.ndim != 2:
        raise ValueError(
            f"S_opt must be 2-D — got {S_opt.ndim}-D with shape {S_opt.shape}."
        )
    if not isinstance(lam_actual, (int, float)):
        raise TypeError(
            f"lam_actual must be numeric — got {type(lam_actual).__name__}."
        )
    if lam_actual < 0:
        raise ValueError(f"lam_actual must be ≥ 0 — got {lam_actual}.")

    # Step 1: Magnitude
    A = np.abs(S_opt)

    # Step 2: Garrote shrinkage ratio  G = max(1 − (λ/A)², 0)
    G = np.where(
        A > lam_actual,
        1.0 - (lam_actual / A) ** 2,   # shrinkage ratio ∈ (0, 1)
        0.0                             # noise → zero
    )

    # Step 3: Denoised complex output (phase preserved)
    S_clean = S_opt * G

    return S_clean, G


In [32]:
## Cell 5: Partial Derivatives

##Central finite-difference partial derivatives ∂M_p/∂θ of the Concentration Measure with respect to each hybrid window parameter.


In [33]:
"""
Partial Derivatives of the Concentration Measure
=================================================
Computes ∂M_p/∂θ — the partial derivative of the Stankovic
Concentration Measure with respect to each hybrid window parameter:

    θ ∈ { γ_GS, γ_HY_B, γ_HY_F, λ² }

Uses central finite differences:
    ∂M_p/∂θ ≈ [ M_p(θ+δ) − M_p(θ−δ) ] / (2δ)

Uses the adaptive S-Transform S_x^a(t, f) which selects the optimal
p for each frequency via CM maximization (Eqs. 7–9).

Hybrid window uses multiplication:
    w_Hybrid[m,k] = w_GS[m,k] × w_HY[m,k]

All parameter values and functions are linked to:
    - Hybrid Window S Transform.py  (adaptive S-Transform computation)
    - Concentration Measure.py      (M_p computation)
"""

import numpy as np

# Functions adaptive_s_transform and concentration_measure
# are defined in earlier cells and available in global scope.


# ============================================================
# 1. SINGLE-PARAMETER PARTIAL DERIVATIVE  ∂M_p/∂θ
# ============================================================
def dMp_dtheta(signal, param_name, gamma_GS, gamma_HY_B, gamma_HY_F,
               lambda_sq, p, delta_frac):
    """
    Central finite-difference partial derivative of M_p w.r.t. one parameter.

    Uses the adaptive S-Transform S_x^a(t, f) which internally performs
    per-frequency p-optimization (Eqs. 7–9).

    Parameters
    ----------
    signal : array-like
        1-D input signal.
    param_name : str
        One of 'gamma_GS', 'gamma_HY_B', 'gamma_HY_F', 'lambda_sq'.
    p : int or float
        Order of the concentration measure (default 2).
    delta_frac : float
        Fractional perturbation size (default 1e-4).
    gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq :
        Baseline parameter values.

    Returns
    -------
    dict with keys:
        'dMp'     : float — ∂M_p/∂θ
        'Mp_plus' : float — M_p(θ + δ)
        'Mp_minus': float — M_p(θ − δ)
        'delta'   : float — absolute perturbation used
    """
    base_params = {
        'gamma_GS':   gamma_GS,
        'gamma_HY_B': gamma_HY_B,
        'gamma_HY_F': gamma_HY_F,
        'lambda_sq':  lambda_sq,
    }

    if param_name not in base_params:
        raise ValueError(
            f"Unknown parameter '{param_name}'. "
            f"Must be one of {list(base_params.keys())}."
        )

    theta_0 = base_params[param_name]
    delta   = max(abs(theta_0) * delta_frac, 1e-8)

    # --- θ + δ ---
    params_plus = base_params.copy()
    params_plus[param_name] = theta_0 + delta

    result_plus = adaptive_s_transform(signal, **params_plus)
    Mp_plus = concentration_measure(result_plus['S_hybrid'], p=p)

    # --- θ − δ ---
    params_minus = base_params.copy()
    params_minus[param_name] = theta_0 - delta
    # Ensure positivity
    params_minus[param_name] = max(params_minus[param_name], 1e-10)

    result_minus = adaptive_s_transform(signal, **params_minus)
    Mp_minus = concentration_measure(result_minus['S_hybrid'], p=p)

    # Central difference
    effective_delta = (params_plus[param_name] - params_minus[param_name]) / 2.0
    dMp = (Mp_plus - Mp_minus) / (2.0 * effective_delta)

    return {
        'dMp':      dMp,
        'Mp_plus':  Mp_plus,
        'Mp_minus': Mp_minus,
        'delta':    effective_delta,
    }


# ============================================================
# 2. ALL PARTIAL DERIVATIVES  ∇_θ M_p
# ============================================================
def compute_concentration_gradients(signal, gamma_GS, gamma_HY_B,
                                    gamma_HY_F, lambda_sq, cm_order, delta_frac):
    """
    Computes ∂M_p/∂θ for every parameter at the given CM order.

    Uses the adaptive S-Transform S_x^a(t, f) for all evaluations.

    Parameters
    ----------
    signal : array-like
        1-D input signal.
    cm_order : int or float
        Order p for the concentration measure (passed from
        the centralized definition in Hybrid Window S Transform.py).
    delta_frac : float
        Fractional perturbation (default 1e-4).
    gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq :
        Baseline parameter values.

    Returns
    -------
    dict with keys:
        'gradients'  : dict[param_name] → float  (∂M_p/∂θ)
        'Mp_base'    : float                       (M_p at baseline)
        'parameters' : dict of baseline parameter values
        'cm_order'   : int or float — the order used
    """
    param_names = ['gamma_GS', 'gamma_HY_B', 'gamma_HY_F', 'lambda_sq']
    base_params = {
        'gamma_GS':   gamma_GS,
        'gamma_HY_B': gamma_HY_B,
        'gamma_HY_F': gamma_HY_F,
        'lambda_sq':  lambda_sq,
    }

    # Baseline M_p value
    result_base = adaptive_s_transform(signal, **base_params)
    Mp_base = concentration_measure(result_base['S_hybrid'], p=cm_order)

    # Gradients
    gradients = {}
    for name in param_names:
        info = dMp_dtheta(
            signal, name, p=cm_order, delta_frac=delta_frac,
            **base_params
        )
        gradients[name] = info['dMp']

    return {
        'gradients':  gradients,
        'Mp_base':    Mp_base,
        'parameters': base_params,
        'cm_order':   cm_order,
    }


# ============================================================
# 3. PRETTY-PRINT GRADIENT TABLE
# ============================================================
_PARAM_SYMBOLS = {
    'gamma_GS':   'γ_GS',
    'gamma_HY_B': 'γ_HY_B',
    'gamma_HY_F': 'γ_HY_F',
    'lambda_sq':  'λ²',
}


def print_concentration_gradients(result):
    """
    Prints the ∂M_p/∂θ table with baseline M_p value.

    Parameters
    ----------
    result : dict
        Output of compute_concentration_gradients().
    """
    cm_order  = result['cm_order']
    gradients = result['gradients']
    Mp_base   = result['Mp_base']
    params    = result['parameters']

    # Header
    print("\n" + "=" * 62)
    print("  ∂M_p/∂θ — Concentration Measure Sensitivity to Window Parameters")
    print("=" * 62)

    # Baseline values
    print(f"\n  Baseline parameters:")
    for name, val in params.items():
        sym = _PARAM_SYMBOLS.get(name, name)
        print(f"    {sym:>8s} = {val}")
    print(f"\n  M(p={cm_order}) at baseline: {Mp_base:.6f}")

    # Gradient table
    col_w = 16
    print("\n" + "-" * 62)
    header = f"  {'Parameter':<12s}  {'∂M(p=' + str(cm_order) + ')/∂θ':<{col_w}s}"
    print(header)
    print("-" * 62)

    for name in ['gamma_GS', 'gamma_HY_B', 'gamma_HY_F', 'lambda_sq']:
        sym = _PARAM_SYMBOLS[name]
        val = gradients[name]
        print(f"  {sym:<12s}  {val:<{col_w}.6f}")

    print("-" * 62)

    # Direction guide
    print("\n  Interpretation:")
    print("    Negative ∂M_p/∂θ → increasing θ improves concentration (lower M_p)")
    print("    Positive ∂M_p/∂θ → increasing θ worsens concentration (higher M_p)")
    print("    Larger |∂M_p/∂θ| → parameter has stronger effect on concentration")
    print("=" * 62)


In [34]:
## Cell 6: Parameter Optimizer

##Normalized gradient descent optimization of hybrid window parameters (γ_GS, γ_HY_B, γ_HY_F, λ²) using the concentration measure as objective.


In [35]:
"""
Gradient Descent Optimizer for Hybrid Window Parameters
=======================================================
Iteratively updates the four hybrid window parameters using the
gradient of the concentration measure H = M_p(S):

    θ_{m+1} = θ_m − μ · (∂H / ∂θ)

where θ ∈ { γ_GS, γ_HY_B, γ_HY_F, λ² }

Uses the adaptive S-Transform S_x^a(t, f) which selects the optimal
p for each frequency via CM maximization (Eqs. 7–9).

Hybrid window uses multiplication:
    w_Hybrid[m,k] = w_GS[m,k] × w_HY[m,k]

Convergence criterion: stop when ΔH = |H_m − H_{m-1}| < tolerance
or when the maximum number of iterations is reached.

Links to:
    - Hybrid Window S Transform.py  (adaptive S-Transform computation)
    - Concentration Measure.py      (H = M_p objective)
    - Partial Derivatives.py        (∂H/∂θ gradient computation)
"""

import numpy as np

# Functions adaptive_s_transform, concentration_measure, and dMp_dtheta
# are defined in earlier cells and available in global scope.


# ============================================================
# Parameter names to optimize (the four window shape params)
# ============================================================
OPTIMIZABLE_PARAMS = ['gamma_GS', 'gamma_HY_B', 'gamma_HY_F', 'lambda_sq']

PARAM_SYMBOLS = {
    'gamma_GS':   'γ_GS',
    'gamma_HY_B': 'γ_HY_B',
    'gamma_HY_F': 'γ_HY_F',
    'lambda_sq':  'λ²',
}

# Minimum allowed values (all must stay positive)
PARAM_MIN = {
    'gamma_GS':   0.01,
    'gamma_HY_B': 0.01,
    'gamma_HY_F': 0.01,
    'lambda_sq':  0.01,
}


# ============================================================
# 1. SINGLE GRADIENT-DESCENT STEP (with gradient normalization)
# ============================================================
def gradient_step(signal, params, mu, p, delta_frac):
    """
    Performs one normalized gradient-descent update:
        g  = [∂H/∂γ_GS, ∂H/∂γ_HY_B, ∂H/∂γ_HY_F, ∂H/∂λ²]
        ĝ  = g / ‖g‖₂           (unit direction)
        θ_{m+1} = θ_m − μ · ĝ   (fixed-magnitude step)

    Uses the adaptive S-Transform S_x^a(t, f) for all evaluations.

    Normalizing by ‖g‖₂ ensures that μ controls the actual step
    size in parameter space regardless of the p-order scale.
    This prevents gradient explosion at higher p (e.g. p=4).

    Parameters
    ----------
    signal : ndarray
        1-D input signal.
    params : dict
        Current parameter values {'gamma_GS': ..., 'gamma_HY_B': ..., ...}.
    mu : float
        Step size (learning rate) — now directly equals the
        Euclidean step length in parameter space.
    p : int or float
        Order of the concentration measure.
    delta_frac : float
        Fractional perturbation for finite-difference gradients.

    Returns
    -------
    dict with keys:
        'new_params' : dict — updated parameter values
        'gradients'  : dict — raw ∂H/∂θ for each parameter
        'H_current'  : float — concentration measure at current params
        'grad_norm'  : float — ‖g‖₂ (gradient magnitude)
    """
    # Compute current H
    result = adaptive_s_transform(signal, **params)
    H_current = concentration_measure(result['S_hybrid'], p=p)

    # Compute raw gradients
    gradients = {}
    for name in OPTIMIZABLE_PARAMS:
        info = dMp_dtheta(
            signal, name, p=p, delta_frac=delta_frac,
            **params
        )
        gradients[name] = info['dMp']

    # Compute L2 norm of the gradient vector
    grad_vec = np.array([gradients[name] for name in OPTIMIZABLE_PARAMS])
    grad_norm = np.linalg.norm(grad_vec)

    # Normalize: step in the gradient direction with magnitude μ
    new_params = {}
    if grad_norm > 1e-30:
        for name in OPTIMIZABLE_PARAMS:
            normalized_grad = gradients[name] / grad_norm
            updated = params[name] - mu * normalized_grad
            new_params[name] = max(updated, PARAM_MIN[name])
    else:
        # Gradient is essentially zero — no update
        new_params = params.copy()

    return {
        'new_params': new_params,
        'gradients':  gradients,
        'H_current':  H_current,
        'grad_norm':  grad_norm,
    }


# ============================================================
# 2. FULL OPTIMIZATION LOOP
# ============================================================
def optimize_parameters(signal, gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq,
                        mu, p, max_iter, tol, delta_frac, verbose):
    """
    Normalized gradient-descent optimization of hybrid window parameters.

    Uses the adaptive S-Transform S_x^a(t, f) for all evaluations.

    Iterates until convergence:
        ĝ  = ∇H / ‖∇H‖₂          (unit gradient direction)
        θ_{m+1} = θ_m − μ · ĝ     (fixed-length step)
        Stop when |H_m − H_{m-1}| < tol  or  m >= max_iter

    Parameters
    ----------
    signal : array-like
        1-D input signal.
    mu : float
        Step size / learning rate (default 0.04).
    p : int or float
        Order of the concentration measure (default 2).
    max_iter : int
        Maximum number of iterations (default 50).
    tol : float
        Convergence tolerance on ΔH (default 1e-3).
    delta_frac : float
        Fractional perturbation for gradient computation.
    gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq :
        Initial parameter values.
    verbose : bool
        If True, print iteration-by-iteration progress.

    Returns
    -------
    dict with keys:
        'optimized_params' : dict — final optimized parameter values
        'initial_params'   : dict — starting parameter values
        'H_initial'        : float — initial concentration measure
        'H_final'          : float — final concentration measure
        'H_history'        : list of float — H at each iteration
        'param_history'    : list of dict — params at each iteration
        'n_iterations'     : int — number of iterations performed
        'converged'        : bool — whether convergence was achieved
    """
    signal = np.asarray(signal, dtype=np.float64)

    # Initial parameter vector
    params = {
        'gamma_GS':   gamma_GS,
        'gamma_HY_B': gamma_HY_B,
        'gamma_HY_F': gamma_HY_F,
        'lambda_sq':  lambda_sq,
    }
    initial_params = params.copy()

    # Compute initial H
    result_0 = adaptive_s_transform(signal, **params)
    H_prev = concentration_measure(result_0['S_hybrid'], p=p)
    H_initial = H_prev

    H_history = [H_prev]
    param_history = [params.copy()]

    if verbose:
        print("\n" + "=" * 78)
        print("  Gradient Descent Optimization of Hybrid Window Parameters")
        print("=" * 78)
        print(f"  Objective: H = M(p={p})    Step size: μ = {mu}")
        print(f"  Tolerance: ΔH < {tol}      Max iterations: {max_iter}")
        print(f"  Hybrid window: w_GS × w_HY (multiplicative)")
        print()
        print(f"  Initial parameters:")
        for name in OPTIMIZABLE_PARAMS:
            sym = PARAM_SYMBOLS[name]
            print(f"    {sym:>8s} = {params[name]:.6f}")
        print(f"  Initial H = {H_initial:.6f}")
        print()

        # Table header
        print("-" * 78)
        hdr = f"  {'Iter':>4s}  {'H':>14s}  {'ΔH':>12s}"
        for name in OPTIMIZABLE_PARAMS:
            hdr += f"  {PARAM_SYMBOLS[name]:>8s}"
        print(hdr)
        print("-" * 78)

    converged = False
    mu_min = 1e-8  # floor to prevent μ from vanishing

    for m in range(1, max_iter + 1):
        # One gradient step
        step = gradient_step(
            signal, params, mu, p=p, delta_frac=delta_frac
        )

        new_params = step['new_params']

        # Compute H at new params to get actual ΔH
        result_new = adaptive_s_transform(signal, **new_params)
        H_new = concentration_measure(result_new['S_hybrid'], p=p)

        delta_H = H_new - H_prev

        # Safeguard: if H increased, revert params and halve μ
        if delta_H > 0:
            mu = max(mu * 0.5, mu_min)
            if verbose:
                row = f"  {m:4d}  {H_new:14.6f}  {delta_H:+12.6f}"
                for name in OPTIMIZABLE_PARAMS:
                    row += f"  {params[name]:8.4f}"
                print(row)
                print(f"  ⚠ H increased — reverting params, reducing μ to {mu:.6f}")
            # Do NOT update params or H_prev — retry with smaller step
            H_history.append(H_prev)
            param_history.append(params.copy())
        else:
            # Accept the step
            params = new_params
            H_prev = H_new

            H_history.append(H_new)
            param_history.append(params.copy())

            if verbose:
                row = f"  {m:4d}  {H_new:14.6f}  {delta_H:+12.6f}"
                for name in OPTIMIZABLE_PARAMS:
                    row += f"  {params[name]:8.4f}"
                print(row)

        # Convergence check: H stopped decreasing
        if abs(delta_H) < tol:
            converged = True
            if verbose:
                print("-" * 78)
                print(f"  ✓ Converged at iteration {m}  (|ΔH| = {abs(delta_H):.8f} < {tol})")
            break

        # Also converge if μ has become negligibly small
        if mu <= mu_min:
            converged = True
            if verbose:
                print("-" * 78)
                print(f"  ✓ Converged at iteration {m}  (μ reached minimum {mu_min})")
            break

    else:
        if verbose:
            print("-" * 78)
            print(f"  ✗ Max iterations ({max_iter}) reached without convergence")

    H_final = H_history[-1]

    if verbose:
        print()
        print(f"  Optimization Summary")
        print(f"  {'─' * 40}")
        print(f"  H_initial = {H_initial:.6f}")
        print(f"  H_final   = {H_final:.6f}")
        improvement = H_initial - H_final
        pct = (improvement / H_initial) * 100 if H_initial != 0 else 0
        print(f"  ΔH_total  = {-improvement:+.6f}  ({pct:+.4f}%)")
        print()
        print(f"  {'Parameter':<12s}  {'Initial':>10s}  {'Optimized':>10s}  {'Change':>10s}")
        print(f"  {'─' * 48}")
        for name in OPTIMIZABLE_PARAMS:
            sym = PARAM_SYMBOLS[name]
            init_val = initial_params[name]
            opt_val  = params[name]
            change   = opt_val - init_val
            print(f"  {sym:<12s}  {init_val:10.6f}  {opt_val:10.6f}  {change:+10.6f}")
        print("=" * 78)

    return {
        'optimized_params': params,
        'initial_params':   initial_params,
        'H_initial':        H_initial,
        'H_final':          H_final,
        'H_history':        H_history,
        'param_history':    param_history,
        'n_iterations':     len(H_history) - 1,
        'converged':        converged,
    }


In [36]:
## Cell 7: DOA Estimation

#ULA signal simulation, per-element S-Transform denoising, spatial covariance, spatial smoothing, ROOT-MUSIC, TLS-ESPIRIT, DML, SPICE, and MVDR algorithms.


In [37]:
def simulate_ula_signals(source_signals, doas_deg, n_sensors,
                         d_over_lambda=0.5, snr_db=15):
    """Simulates ULA received signals."""
    D, N = source_signals.shape
    M = n_sensors
    doas_rad = np.radians(doas_deg)
    m_idx = np.arange(M).reshape(-1, 1)
    phases = 2 * np.pi * d_over_lambda * np.sin(doas_rad)
    A_steer = np.exp(1j * m_idx * phases)
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        X_clean = A_steer @ source_signals
    signal_power = np.mean(np.abs(X_clean)**2)
    noise_power = signal_power / (10**(snr_db / 10))
    noise = np.sqrt(noise_power / 2) * (np.random.randn(M, N) + 1j * np.random.randn(M, N))
    return X_clean + noise, A_steer


def denoise_array_elements(X_array, gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq,
                           skip_optimization, cm_order, mu=0.04, keep_freq_bins=None):
    """S-Transform + Denoising per ULA element."""
    M, N = X_array.shape
    S_clean_list, S_opt_list, sigmas, lams = [], [], [], []
    st_params = dict(gamma_GS=gamma_GS, gamma_HY_B=gamma_HY_B,
                     gamma_HY_F=gamma_HY_F, lambda_sq=lambda_sq)

    for m in range(M):
        x_m = X_array[m, :]
        if skip_optimization:
            res = adaptive_s_transform(x_m, **st_params)
        else:
            opt_res = optimize_parameters(x_m, gamma_GS, gamma_HY_B, gamma_HY_F, lambda_sq,
                                          mu=mu, p=cm_order, max_iter=50, tol=1e-3,
                                          delta_frac=1e-4, verbose=False)
            res = adaptive_s_transform(x_m, **opt_res["optimized_params"])
        S_opt_m = res["S_hybrid"]
        A_m = extract_magnitude(S_opt_m)
        sigma_m = estimate_noise_variance(A_m)
        lam_m, _, _ = actual_threshold(sigma_m, A_m.shape[1])
        S_clean_m, G_m = apply_mask(S_opt_m, lam_m)
        del A_m, G_m

        if keep_freq_bins is not None:
            S_clean_list.append(S_clean_m[keep_freq_bins, :])
            del S_opt_m, S_clean_m
        else:
            S_opt_list.append(S_opt_m)
            S_clean_list.append(S_clean_m)
        sigmas.append(sigma_m)
        lams.append(lam_m)

    return S_clean_list, S_opt_list, {
        "sigma_mean": np.mean(sigmas), "sigma_all": sigmas,
        "lam_mean": np.mean(lams), "lam_all": lams,
    }


def build_spatial_covariance(S_clean_list, freq_bins=None):
    """Builds spatial covariance from denoised TFRs."""
    M = len(S_clean_list)
    n_freqs = S_clean_list[0].shape[0]
    n_times = S_clean_list[0].shape[1]
    if freq_bins is None:
        freq_bins = np.arange(1, n_freqs)
    cube = np.stack(S_clean_list, axis=0)
    cube_sel = cube if n_freqs == len(freq_bins) else cube[:, freq_bins, :]
    L = cube_sel.shape[1] * cube_sel.shape[2]
    Z = cube_sel.reshape(M, L)
    nonzero_mask = np.any(Z != 0, axis=0)
    Z_active = Z[:, nonzero_mask]
    L_active = Z_active.shape[1]
    if L_active == 0:
        return np.zeros((M, M), dtype=complex), 0
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        R = (Z_active @ Z_active.conj().T) / L_active
    J = np.eye(M)[::-1]
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        R = 0.5 * (R + J @ R.conj() @ J)
    return R, L_active


def spatial_smoothing(R, subarray_size, fb=True):
    """Forward-backward spatial smoothing."""
    M = R.shape[0]
    L, K = subarray_size, M - subarray_size + 1
    R_smooth = np.zeros((L, L), dtype=complex)
    for i in range(K):
        R_smooth += R[i:i+L, i:i+L]
    R_smooth /= K
    if fb:
        J = np.eye(L)[::-1]
        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            R_smooth = 0.5 * (R_smooth + J @ R_smooth.conj() @ J)
    return R_smooth, K


def root_music(R, n_sources, array_spacing_lambda=0.5):
    """ROOT-MUSIC DOA estimator."""
    M = R.shape[0]
    eigenvalues, eigenvectors = np.linalg.eigh(R)
    E_n = eigenvectors[:, :M - n_sources]
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        C_mat = E_n @ E_n.conj().T
    poly_coeffs = np.zeros(2*M - 1, dtype=complex)
    for k in range(-(M-1), M):
        poly_coeffs[k + (M-1)] = np.sum(np.diag(C_mat, k))
    all_roots = np.roots(poly_coeffs[::-1])

    inside_mask = np.abs(all_roots) <= 1.0 + 1e-6
    inside_roots = all_roots[inside_mask]
    if len(inside_roots) < n_sources:
        inside_roots = all_roots

    dist_to_unit = np.abs(np.abs(inside_roots) - 1.0)
    chosen_roots = inside_roots[np.argsort(dist_to_unit)[:n_sources]]
    sin_theta = np.clip(np.angle(chosen_roots) / (2*np.pi*array_spacing_lambda), -1.0, 1.0)
    doa_deg = np.sort(np.degrees(np.arcsin(sin_theta)))
    return doa_deg, chosen_roots, all_roots, eigenvalues


def music_spectrum(R, n_sources, n_angles=361, array_spacing_lambda=0.5):
    """Classic MUSIC pseudo-spectrum."""
    M = R.shape[0]
    eigenvalues, eigenvectors = np.linalg.eigh(R)
    E_n = eigenvectors[:, :M - n_sources]
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        En_proj = E_n @ E_n.conj().T
    angles_deg = np.linspace(-90, 90, n_angles)
    P_music = np.zeros(n_angles)
    for i, ang in enumerate(angles_deg):
        a = np.exp(1j * 2*np.pi*array_spacing_lambda * np.sin(np.radians(ang)) * np.arange(M))
        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            denom = (a.conj().T @ En_proj @ a).real
        P_music[i] = 1.0 / (denom + 1e-30)
    return angles_deg, P_music


def tls_espirit(R, n_sources, array_spacing_lambda=0.5):
    """TLS-ESPRIT DOA estimator."""
    M = R.shape[0]
    eigenvalues, eigenvectors = np.linalg.eigh(R)
    E_s = eigenvectors[:, -n_sources:]
    E_12 = np.hstack((E_s[:-1, :], E_s[1:, :]))
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        U, S, Vh = np.linalg.svd(E_12, full_matrices=False)
    V = Vh.conj().T
    V12 = V[:n_sources, n_sources:2*n_sources]
    V22 = V[n_sources:2*n_sources, n_sources:2*n_sources]
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        Phi = -V12 @ np.linalg.inv(V22)
    phi_eigvals = np.linalg.eigvals(Phi)
    sin_theta = np.clip(np.angle(phi_eigvals) / (2*np.pi*array_spacing_lambda), -1.0, 1.0)
    return np.sort(np.degrees(np.arcsin(sin_theta))), phi_eigvals, eigenvalues


def _steering_matrix(thetas_rad, M, d_over_lambda):
    """ULA steering matrix."""
    m_idx = np.arange(M).reshape(-1, 1)
    return np.exp(1j * m_idx * 2*np.pi*d_over_lambda*np.sin(thetas_rad))


def _dml_cost(thetas_rad, R, M, d_over_lambda):
    """DML cost function."""
    A = _steering_matrix(thetas_rad, M, d_over_lambda)
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        AHA = A.conj().T @ A
    try:
        AHA_inv = np.linalg.inv(AHA)
    except np.linalg.LinAlgError:
        return 1e12
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        P_A = A @ AHA_inv @ A.conj().T
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        return np.real(np.trace((np.eye(M) - P_A) @ R))


def deterministic_ml(R, n_sources, array_spacing_lambda=0.5,
                     grid_points=361, refine=True, verbose=False):
    """Deterministic Maximum Likelihood DOA estimator."""
    from scipy.optimize import minimize as sp_minimize
    M = R.shape[0]
    d_lam = array_spacing_lambda

    try:
        music_doa, _, _, _ = root_music(R, n_sources, array_spacing_lambda=d_lam)
    except Exception:
        music_doa = np.linspace(-60, 60, n_sources)

    init_angles_deg = np.copy(music_doa)
    cost_init = _dml_cost(np.radians(init_angles_deg), R, M, d_lam)
    if verbose:
        print(f"    DML init (MUSIC): {np.round(init_angles_deg, 3).tolist()}°  cost={cost_init:.6f}")

    info = {'init_angles_deg': init_angles_deg, 'n_iterations': 0,
            'converged': False, 'cost_init': cost_init, 'cost_final': cost_init}

    if not refine:
        return np.sort(init_angles_deg), cost_init, info

    try:
        result = sp_minimize(
            lambda x: _dml_cost(np.radians(x), R, M, d_lam),
            x0=init_angles_deg, method='L-BFGS-B',
            bounds=[(-89.0, 89.0)] * n_sources,
            options={'maxiter': 200, 'ftol': 1e-12, 'gtol': 1e-8}
        )
        info.update({'n_iterations': result.nit, 'converged': result.success, 'cost_final': result.fun})
        if verbose:
            status = "✓ converged" if result.success else "⚠ did not converge"
            print(f"    DML optimised: {np.round(np.sort(result.x), 4).tolist()}°  cost={result.fun:.6f}  ({status})")
        return np.sort(result.x), result.fun, info
    except Exception:
        return np.sort(init_angles_deg), cost_init, info


def spice(R, n_sources, array_spacing_lambda=0.5,
          n_grid=361, max_iter=200, tol=1e-6, verbose=False):
    """SPICE DOA estimator."""
    M = R.shape[0]
    d_lam = array_spacing_lambda
    grid_deg = np.linspace(-90, 90, n_grid)
    K = n_grid
    A = _steering_matrix(np.radians(grid_deg), M, d_lam)

    R_trace = np.real(np.trace(R))
    scale = R_trace / M if R_trace > 0 else 1.0
    R_hat = (0.5 * (R + R.conj().T)) / scale

    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        AH_R = A.conj().T @ R_hat
    matched = np.real(np.sum(AH_R * A.conj().T, axis=1))
    p_vec = np.maximum(matched / (M*M), 1e-20)
    sigma2 = np.real(np.trace(R_hat)) / (2.0*M)
    converged, n_iter = False, 0

    for it in range(max_iter):
        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            R_model = (A * p_vec[np.newaxis, :]) @ A.conj().T + sigma2 * np.eye(M)
        try:
            W = np.linalg.inv(R_model)
        except np.linalg.LinAlgError:
            W = np.linalg.inv(R_model + 1e-10*np.eye(M))

        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            WA = W @ A
            WRW = W @ R_hat @ W
        norm_Wa = np.sqrt(np.real(np.sum(WA.conj() * WA, axis=0)))
        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            aH_WRW_a = np.real(np.sum(A.conj() * (WRW @ A), axis=0))
        denom = np.sqrt(np.maximum(aH_WRW_a, 1e-30))
        p_new = p_vec * norm_Wa / denom

        norm_W_F = np.sqrt(np.real(np.sum(W.conj() * W)))
        sigma2_new = sigma2 * norm_W_F / np.sqrt(max(M*np.real(np.trace(WRW)), 1e-30))

        total_power = np.sum(p_new) + M*sigma2_new
        if total_power > 0:
            nf = np.real(np.trace(R_hat)) / total_power
            p_new *= nf
            sigma2_new *= nf

        rel_change = np.linalg.norm(p_new - p_vec) / (np.linalg.norm(p_vec) + 1e-30)
        p_vec = np.maximum(p_new, 1e-30)
        sigma2 = max(sigma2_new, 1e-30)
        n_iter = it + 1
        if rel_change < tol:
            converged = True
            break

    if verbose:
        print(f"    SPICE {'✓ converged' if converged else '⚠ max iters'} after {n_iter} iterations")

    p_db = 10 * np.log10(p_vec / np.max(p_vec) + 1e-30)
    peak_indices = [i for i in range(1, K-1) if p_vec[i] > p_vec[i-1] and p_vec[i] > p_vec[i+1]]
    if not peak_indices:
        peak_indices = list(np.argsort(p_vec)[-n_sources:])
    peak_indices = np.array(peak_indices)

    strong = p_db[peak_indices] > -20.0
    if np.sum(strong) >= n_sources:
        peak_indices = peak_indices[strong]
    if len(peak_indices) > n_sources:
        peak_indices = peak_indices[np.argsort(p_vec[peak_indices])[-n_sources:]]
    if len(peak_indices) < n_sources:
        remaining = np.setdiff1d(np.arange(K), peak_indices)
        peak_indices = np.concatenate([peak_indices, remaining[np.argsort(p_vec[remaining])[::-1]][:n_sources - len(peak_indices)]])
    peak_indices = np.sort(peak_indices)

    doa_estimates = []
    for idx in peak_indices:
        if 0 < idx < K-1:
            a, b, g = p_db[idx-1], p_db[idx], p_db[idx+1]
            d_interp = a - 2*b + g
            if abs(d_interp) > 1e-12:
                delta = 0.5*(a - g)/d_interp
                doa_estimates.append(np.clip(grid_deg[idx] + delta*(grid_deg[1]-grid_deg[0]), -90, 90))
            else:
                doa_estimates.append(grid_deg[idx])
        else:
            doa_estimates.append(grid_deg[idx])

    return np.sort(np.array(doa_estimates)), p_vec, {
        'grid_deg': grid_deg, 'n_iterations': n_iter, 'converged': converged,
        'sigma2': sigma2, 'peak_indices': peak_indices, 'peak_powers_db': p_db[peak_indices]
    }


def mvdr(R, n_sources, array_spacing_lambda=0.5,
         n_grid=361, diagonal_loading=1e-6, verbose=False):
    """MVDR (Capon) DOA estimator."""
    M = R.shape[0]
    d_lam = array_spacing_lambda
    R_loaded = R + diagonal_loading*np.eye(M)
    cond_num = np.linalg.cond(R_loaded)
    try:
        R_inv = np.linalg.inv(R_loaded)
    except np.linalg.LinAlgError:
        R_inv = np.linalg.inv(R + 1e-3*np.eye(M))

    grid_deg = np.linspace(-90, 90, n_grid)
    K = n_grid
    A = _steering_matrix(np.radians(grid_deg), M, d_lam)
    with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
        R_inv_A = R_inv @ A
    denom_vec = np.maximum(np.real(np.sum(A.conj() * R_inv_A, axis=0)), 1e-30)
    P_mvdr = 1.0 / denom_vec
    P_mvdr_db = 10*np.log10(P_mvdr / np.max(P_mvdr) + 1e-30)

    if verbose:
        print(f"    MVDR: cond={cond_num:.2e}  dynamic_range={np.max(P_mvdr_db)-np.min(P_mvdr_db):.1f} dB")

    peak_indices = [i for i in range(1, K-1) if P_mvdr[i] > P_mvdr[i-1] and P_mvdr[i] > P_mvdr[i+1]]
    if not peak_indices:
        peak_indices = list(np.argsort(P_mvdr)[-n_sources:])
    peak_indices = np.array(peak_indices)

    strong = P_mvdr_db[peak_indices] > -15.0
    if np.sum(strong) >= n_sources:
        peak_indices = peak_indices[strong]
    if len(peak_indices) > n_sources:
        peak_indices = peak_indices[np.argsort(P_mvdr[peak_indices])[-n_sources:]]
    if len(peak_indices) < n_sources:
        remaining = np.setdiff1d(np.arange(K), peak_indices)
        peak_indices = np.concatenate([peak_indices, remaining[np.argsort(P_mvdr[remaining])[::-1]][:n_sources - len(peak_indices)]])
    peak_indices = np.sort(peak_indices)

    doa_estimates = []
    for idx in peak_indices:
        if 0 < idx < K-1:
            a, b, g = P_mvdr_db[idx-1], P_mvdr_db[idx], P_mvdr_db[idx+1]
            d_interp = a - 2*b + g
            if abs(d_interp) > 1e-12:
                delta = 0.5*(a - g)/d_interp
                doa_estimates.append(np.clip(grid_deg[idx] + delta*(grid_deg[1]-grid_deg[0]), -90, 90))
            else:
                doa_estimates.append(grid_deg[idx])
        else:
            doa_estimates.append(grid_deg[idx])

    return np.sort(np.array(doa_estimates)), P_mvdr, {
        'grid_deg': grid_deg, 'P_mvdr_db': P_mvdr_db,
        'peak_indices': peak_indices, 'peak_powers_db': P_mvdr_db[peak_indices],
        'condition_number': cond_num,
    }


In [38]:

# Pipeline Execution

## Section 1: Load Real THz Dataset (SigMF)


In [39]:
# ============================================================
# 1. LOAD REAL THz DATASET (SigMF)
# ============================================================
print("=" * 65)
print("  THz Signal Processing — Real NIST Dataset")
print("  Hybrid Window S-Transform Pipeline")
print("=" * 65)


# --- Dataset path ---
# ⚠️ UPDATE THIS PATH to match your Kaggle dataset location
# Upload your dataset to Kaggle and set the path accordingly
DATASET_DIR = os.path.join(
    "/kaggle/input/datasets/abhinabadas8918/thz-dataset/1-1.05 THz/1.02THz_10cm_2PSK_IF_5G_5GSyms_600mV"
)

# --- Discover all frames ---
frames = list_frames(DATASET_DIR)
n_frames = len(frames)
print(f"\n  Dataset directory: {os.path.basename(DATASET_DIR)}")
print(f"  Frames found    : {n_frames}")

# --- Load Frame 1 as the primary signal ---
frame1 = load_sigmf(frames[0]['data_path'])

# --- Extract metadata ---
Fs_hz      = frame1['sample_rate']         # 160 GS/s (Hz)
Fs         = Fs_hz / 1e12                  # in THz (for compatibility)
dt         = frame1['dt']                  # 6.25 ps
rf_freq    = frame1['rf_freq']             # ~1.005 THz
lo_freq    = frame1['lo_freq']             # 1.0 THz
if_freq    = frame1['center_freq']         # 5 GHz
full_len   = frame1['n_samples']           # 76,928

print(f"\n  Signal source    : {frame1['description']}")
print(f"  Data type        : {frame1['datatype']}")
print(f"  Samples (total)  : {full_len:,}")
print(f"  Sample rate      : {Fs_hz/1e9:.1f} GS/s")
print(f"  Sampling interval: {dt:.4f} ps")
print(f"  RF carrier       : {rf_freq/1e12:.4f} THz")
print(f"  LO frequency     : {lo_freq/1e12:.4f} THz")
print(f"  IF frequency     : {if_freq/1e9:.1f} GHz")
print(f"  Duration (total) : {frame1['duration_ns']:.3f} ns")
if frame1['link_distance'] is not None:
    print(f"  Link distance    : {frame1['link_distance']*100:.0f} cm")

if frame1['emitter_info']:
    ei = frame1['emitter_info']
    print(f"  Modulation       : BPSK at {ei['symbol_rate']/1e9:.1f} Gsym/s")
    print(f"  TX amplitude     : {ei['amplitude_mV']:.0f} mV")

if frame1['filter_info']:
    fi = frame1['filter_info']
    print(f"  Pulse shaping    : {fi['window_type']} (β={fi['beta']}, "
          f"span={fi['span']}, SPS={fi['sps']})")

# --- Extract a segment for S-Transform processing ---
# 1024 samples = 16 BPSK symbols × 64 samples/symbol
# Centred on the peak energy region of the signal
SEGMENT_LEN = 3000
seg_result  = extract_segment(
    frame1['signal'], n_samples=SEGMENT_LEN,
    center='peak_energy', component='real'
)
signal    = seg_result['segment']        # real-valued 1-D array for S-Transform
N_samples = len(signal)

print(f"\n  Segment extracted:")
print(f"    Length         : {N_samples} samples ({N_samples / (ei['symbol_rate']/Fs_hz * 1):.0f} symbols × {fi['sps']} SPS)" if frame1['filter_info'] else f"    Length         : {N_samples} samples")
print(f"    Window         : [{seg_result['start_idx']}, {seg_result['end_idx']})")
print(f"    Component      : {seg_result['component']} part of IQ")
print(f"    Signal range   : [{signal.min():.6f}, {signal.max():.6f}]")
print(f"    RMS            : {np.sqrt(np.mean(signal**2)):.6f}")

# Build a time axis for the segment (in ps)
time_axis = np.arange(N_samples) * dt


  THz Signal Processing — Real NIST Dataset
  Hybrid Window S-Transform Pipeline

  Dataset directory: 1.02THz_10cm_2PSK_IF_5G_5GSyms_600mV
  Frames found    : 7

  Signal source    : Experimental THz Modulated Data transmitted 10cm at 1.02 THz RF, with 5GHz IF. Frame 1.
  Data type        : ci16_le
  Samples (total)  : 76,928
  Sample rate      : 160.0 GS/s
  Sampling interval: 6.2500 ps
  RF carrier       : 1.0050 THz
  LO frequency     : 1.0000 THz
  IF frequency     : 5.0 GHz
  Duration (total) : 480.800 ns
  Link distance    : 10 cm
  Modulation       : BPSK at 5.0 Gsym/s
  TX amplitude     : 600 mV
  Pulse shaping    : RRC (β=0.75, span=10, SPS=32)

  Segment extracted:
    Length         : 3000 samples (96000 symbols × 32 SPS)
    Window         : [1043, 4043)
    Component      : real part of IQ
    Signal range   : [-0.993286, 0.999573]
    RMS            : 0.601756


In [40]:
## Section 2: Window Parameters & Adaptive S-Transform


In [41]:
# ============================================================
# 2. WINDOW PARAMETERS & ADAPTIVE S-TRANSFORM
# ============================================================

# --- Concentration measure order ---
cm_order = 4

# --- Base Window Parameters (same starting values — optimizer will tune) ---
win_gamma_GS   = 1.0
win_gamma_HY_B = 1.5
win_gamma_HY_F = 0.5
win_lambda_sq  = 10.0
win_mu         = 0.04   # smaller step size for real data stability

print(f"\n  Window parameters:")
print(f"    γ_GS   = {win_gamma_GS}")
print(f"    γ_HY_B = {win_gamma_HY_B}")
print(f"    γ_HY_F = {win_gamma_HY_F}")
print(f"    λ²     = {win_lambda_sq}")
print(f"    μ      = {win_mu}")

# --- Run Adaptive S-Transform ---
print(f"\n  Computing Adaptive S-Transform S_x^a(t, f) ...")
result = adaptive_s_transform(
    signal,
    gamma_GS=win_gamma_GS, gamma_HY_B=win_gamma_HY_B,
    gamma_HY_F=win_gamma_HY_F, lambda_sq=win_lambda_sq
)
print(f"  Done in {result['elapsed_sec']:.3f} s")


S_hyb = result['S_hybrid']

# --- Time resolution at the dominant frequency ---
# The signal is at IF = 5 GHz, sampled at 160 GS/s
# Use the known IF frequency instead of argmax to avoid locking onto
# high-frequency noise (argmax can pick bin 125 ≈ 39 GHz instead of
# the actual signal at bin 16 ≈ 5 GHz).
peak_freq_bin = int(round(if_freq / Fs_hz * N_samples))
peak_freq_Hz  = peak_freq_bin * Fs_hz / N_samples
peak_freq_GHz = peak_freq_Hz / 1e9

print(f"\n  Signal IF mapped to frequency bin {peak_freq_bin} "
      f"(≈ {peak_freq_GHz:.2f} GHz)")

# --- Build comparison table ---
metrics = {}
for label, S in [('Hybrid', S_hyb)]:
    tr = time_resolution(S, peak_freq_bin)
    metrics[label] = {'time_res': tr}

print(f"\n  {'Method':<12s}  {'Time Resolution (samples)'}")
print(f"  {'-'*40}")
for label, m in metrics.items():
    print(f"  {label:<12s}  {m['time_res']}")



  Window parameters:
    γ_GS   = 1.0
    γ_HY_B = 1.5
    γ_HY_F = 0.5
    λ²     = 10.0
    μ      = 0.04

  Computing Adaptive S-Transform S_x^a(t, f) ...
  Done in 5.519 s

  Signal IF mapped to frequency bin 94 (≈ 5.01 GHz)

  Method        Time Resolution (samples)
  ----------------------------------------
  Hybrid        2999


In [42]:
## Section 3: Parameter Optimization

In [ ]:
# ============================================================
# 3. PARAMETER OPTIMIZATION
# ============================================================

print(f"\n{'='*65}")
print(f"  Optimizing window parameters for real THz signal ...")
print(f"{'='*65}")

opt_result = optimize_parameters(
    signal,
    gamma_GS=win_gamma_GS, gamma_HY_B=win_gamma_HY_B,
    gamma_HY_F=win_gamma_HY_F, lambda_sq=win_lambda_sq,
    mu=win_mu,
    p=cm_order,
    max_iter=50,
    tol=1e-3,
    delta_frac=1e-4,
    verbose=True
)

# --- Re-evaluate with optimized parameters ---
opt_params = opt_result['optimized_params']
print(f"\n  Re-running Adaptive S-Transform with optimized parameters ...")
result_opt = adaptive_s_transform(signal, **opt_params)
S_opt = result_opt['S_hybrid']


cm_opt  = concentration_measure(S_opt, p=cm_order)
cm_orig = concentration_measure(S_hyb, p=cm_order)
print(f"\n  M(p={cm_order}) before optimization: {cm_orig:.6f}")
print(f"  M(p={cm_order}) after  optimization: {cm_opt:.6f}")
improvement = cm_orig - cm_opt
pct = (improvement / cm_orig) * 100 if cm_orig != 0 else 0
print(f"  Improvement: {improvement:.6f} ({pct:.4f}%)")



  Optimizing window parameters for real THz signal ...

  Gradient Descent Optimization of Hybrid Window Parameters
  Objective: H = M(p=4)    Step size: μ = 0.04
  Tolerance: ΔH < 0.001      Max iterations: 50
  Hybrid window: w_GS × w_HY (multiplicative)

  Initial parameters:
        γ_GS = 1.000000
      γ_HY_B = 1.500000
      γ_HY_F = 0.500000
          λ² = 10.000000
  Initial H = 49898062813147938816.000000

------------------------------------------------------------------------------
  Iter               H            ΔH      γ_GS    γ_HY_B    γ_HY_F        λ²
------------------------------------------------------------------------------
     1  49682767511341490176.000000  -215295301806448640.000000    0.9892    1.4954    0.5382   10.0000
     2  49409339194771677184.000000  -273428316569812992.000000    0.9751    1.4896    0.5752   10.0000
     3  49394779033702596608.000000  -14560161069080576.000000    0.9405    1.4857    0.5949   10.0000
     4  49233037627218599936.0000

In [ ]:
## Section 4: Garrote Denoising

In [ ]:
# ============================================================
# 4. DENOISING
# ============================================================

A       = extract_magnitude(S_opt)
sigma   = estimate_noise_variance(A)
lam_actual, lam_star, _ = actual_threshold(sigma, A.shape[1])
S_clean, G = apply_mask(S_opt, lam_actual)

print(f"\n  Denoising summary (Garrote):")
print(f"    σ (MAD estimate)       = {sigma:.6f}")
print(f"    Λ* (minimax, σ=1)      = {lam_star:.6f}")
print(f"    λ_actual (σ · Λ*)      = {lam_actual:.6f}")
retained = np.sum(G > 0)
print(f"    Pixels retained        = {retained:.0f} / {G.size}"
      f"  ({100 * retained / G.size:.1f}%)")


In [ ]:
## Section 5: Frame Comparison (Frame 1 vs Frame 2)

In [ ]:
"""# ============================================================
# 5. COMPARE: FRAME 1 vs FRAME 2 (Two Real Captures)
# ============================================================
print(f"\n{'='*65}")
print(f"  Frame Comparison: Frame 1 vs Frame 2 (same setup, different noise)")
print(f"{'='*65}")

# Load Frame 2
frame2 = load_sigmf(frames[1]['data_path'])
seg2   = extract_segment(
    frame2['signal'], n_samples=SEGMENT_LEN,
    center='peak_energy', component='real'
)
signal_f2 = seg2['segment']

# S-Transform of Frame 2 (with same base parameters)
result_f2 = adaptive_s_transform(
    signal_f2,
    gamma_GS=win_gamma_GS, gamma_HY_B=win_gamma_HY_B,
    gamma_HY_F=win_gamma_HY_F, lambda_sq=win_lambda_sq
)
S_f2 = result_f2['S_hybrid']

# Denoise Frame 2
A_f2     = extract_magnitude(S_f2)
sigma_f2 = estimate_noise_variance(A_f2)
lam_f2, _, _ = actual_threshold(sigma_f2, A_f2.size)
S_clean_f2, _ = apply_mask(S_f2, lam_f2)

cm_f1_noisy   = concentration_measure(S_hyb, p=cm_order)
cm_f1_opt     = cm_opt
cm_f1_clean   = concentration_measure(S_clean, p=cm_order)
cm_f2_noisy   = concentration_measure(S_f2, p=cm_order)
cm_f2_clean   = concentration_measure(S_clean_f2, p=cm_order)

print(f"\n  Concentration Measure Comparison:")
print(f"    Frame 1 (before optimization) : M = {cm_f1_noisy:.6f}")
print(f"    Frame 1 (after optimization)  : M = {cm_f1_opt:.6f}")
print(f"    Frame 1 (after denoising)     : M = {cm_f1_clean:.6f}")
print(f"    Frame 2 (before optimization) : M = {cm_f2_noisy:.6f}")
print(f"    Frame 2 (after denoising)     : M = {cm_f2_clean:.6f}")

# Energy preservation check
energy_before = np.sum(np.abs(S_opt)**2)
energy_after  = np.sum(np.abs(S_clean)**2)
print(f"\n  Energy preservation (Frame 1):")
print(f"    Before denoising : {energy_before:.6f}")
print(f"    After denoising  : {energy_after:.6f}")
print(f"    Retained         : {100 * energy_after / energy_before:.1f}%")

# Cross-frame noise consistency
print(f"\n  Noise consistency across frames:")
print(f"    σ (Frame 1) = {sigma:.6f}")
print(f"    σ (Frame 2) = {sigma_f2:.6f}")
print(f"    Ratio       = {sigma / sigma_f2:.4f}  (ideal: 1.0)")

print(f"\n{'='*65}")
print(f"  ✓ THz S-Transform + Denoising pipeline validated on real data")
print(f"{'='*65}")
"""

In [ ]:
## Section 6: DOA Estimation — Configuration

#Change `n_src` and `true_doas_deg` below to configure the DOA estimation.


In [ ]:
# ============================================================
# 6. DOA ESTIMATION via ROOT-MUSIC (THz ULA Simulation)
# ============================================================
# PHYSICS AT THz FREQUENCIES:
#   At f = 1.02 THz → λ = c/f = 3×10⁸ / 1.02×10¹² ≈ 0.294 mm
#   Half-wavelength spacing: d = λ/2 ≈ 147 μm
#   This is physically achievable with micro-fabricated THz antenna arrays.
#   The d_over_lambda ratio stays at 0.5 (dimensionless), same as at any
#   frequency — only the physical size of the array changes.
#
# SOURCE WAVEFORMS:
#   Instead of synthetic material-filtered pulses, we use the 5 real
#   captured frames as 5 independent source signals. Each frame is a
#   separate capture of the same BPSK transmission, with different
#   noise realizations — making them physically distinct waveforms.



# ╔════════════════════════════════════════════════════════════╗
# ║  USER-CONFIGURABLE — change ONLY these two variables      ║
# ║  Everything else adapts automatically.                     ║
# ╚════════════════════════════════════════════════════════════╝
n_src          =6
true_doas_deg  = np.array([-60,-50,-40, -10, 10, 50])


In [ ]:
## Section 7: Adaptive Configuration (auto-derived)

In [ ]:
# ============================================================
# 7. ADAPTIVE CONFIGURATION (auto-derived from the above)
# ============================================================
# --- Validate user inputs ---
assert len(true_doas_deg) == n_src, (
    f"Mismatch: n_src={n_src} but {len(true_doas_deg)} DOA angles given."
)

assert np.all(np.abs(true_doas_deg) <= 90), (
    f"DOA angles must be in [-90°, +90°]."
)

# --- Detect endfire angles (|θ| > 65°) ---
has_endfire = np.any(np.abs(true_doas_deg) > 65)
endfire_max = np.max(np.abs(true_doas_deg))

# --- Auto-compute n_sensors ---
#   Base rule:  M = 4·D + 1  (extra headroom for spatial smoothing,
#   which reduces effective array from M to L = M − D + 1)
#   Endfire:    add extra sensors proportional to cos(θ) degradation
#   NOTE: Do NOT use a fixed floor (e.g. max(45, ...)).  A floor that
#   is too large relative to D creates a polynomial with far too many
#   spurious roots, causing ROOT-MUSIC to fail on root selection.
d_over_lambda  = 0.5  # half-wavelength spacing (fixed physics)
base_sensors   = 4 * n_src + 1
if has_endfire:
    # cos(θ) sensitivity penalty → need more aperture
    endfire_penalty = int(np.ceil(6 / np.cos(np.radians(endfire_max))))
    n_sensors = base_sensors + endfire_penalty
else:
    n_sensors = base_sensors

# --- Auto-compute SNR ---
#   Keep SNR strictly uniform across all angles for academic rigor.
snr_db = 15


# --- Auto-compute frequency band half-width ---
#   Base: ±2 bins (5 total).  Endfire: widen to ±4 bins (9 total).
freq_band_half = 4 if has_endfire else 2

# --- Print adaptive configuration summary ---
print(f"\n\n{'='*65}")
print(f"  DOA Estimation — ROOT-MUSIC with Real THz Source Waveforms")
print(f"{'='*65}")
print(f"  User inputs:")
print(f"    Sources D        : {n_src}")
print(f"    True DOAs        : {true_doas_deg.tolist()} °")
print(f"  Auto-configured:")
print(f"    ULA elements M   : {n_sensors}  (rule: M ≥ 4D+1"
      f"{f', +endfire boost' if has_endfire else ''})")
print(f"    SNR              : {snr_db} dB")
print(f"    Spacing d/λ      : {d_over_lambda}")
print(f"    Freq band half   : ±{freq_band_half} bins")
if has_endfire:
    print(f"    ⚠ Endfire angle detected: {endfire_max:.1f}° "
          f"(cos={np.cos(np.radians(endfire_max)):.3f}, sensitivity reduced)")
print(f"  At 1.02 THz: λ ≈ 0.294 mm, d ≈ 147 μm (micro-fabricated array)")

# --- Minimum angular separation check ---
sorted_angles = np.sort(true_doas_deg)
min_sep = np.min(np.diff(sorted_angles)) if len(sorted_angles) > 1 else 180
# Rayleigh-like resolution limit for a ULA: ~0.886λ/(M·d) in radians
resolution_limit_deg = np.degrees(0.886 / (n_sensors * d_over_lambda))
if min_sep < 2 * resolution_limit_deg:
    print(f"  ⚠ WARNING: Minimum angular separation ({min_sep:.1f}°) is close to "
          f"array resolution limit ({resolution_limit_deg:.1f}°).")
    print(f"    ROOT-MUSIC may struggle to resolve the closest pair.")
else:
    print(f"  ✓ Angular separation OK: min {min_sep:.1f}° "
          f"(resolution limit ≈ {resolution_limit_deg:.1f}°)")


In [ ]:
## Section 7a: Source Waveform Generation


In [ ]:
# ============================================================
# 7a. SOURCE WAVEFORM GENERATION (from main dataset frames)
# ============================================================
# Each frame in the main dataset is a separate capture of the
# same BPSK transmission with different noise realizations —
# making them physically distinct waveforms.  ROOT-MUSIC
# requires uncorrelated sources to separate signal vs noise
# subspace, and staggered segment offsets ensure low
# cross-correlation between source waveforms.
# ============================================================

print(f"\n  Source waveforms: {n_src} frames from main dataset ({n_frames} available)")

# --- Validate that we have enough frames ---
assert n_src <= n_frames, (
    f"n_src={n_src} exceeds available frames in main dataset ({n_frames})."
)

source_signals = np.zeros((n_src, N_samples))
for k in range(n_src):
    frame_k = load_sigmf(frames[k]['data_path'])
    # Extract a different, uncorrelated segment for each source
    # by offsetting the center index. This prevents ROOT-MUSIC from
    # failing due to highly correlated (coherent) source signals.
    center_idx = 5000 + k * 12000
    seg_k   = extract_segment(
        frame_k['signal'], n_samples=SEGMENT_LEN,
        center=center_idx, component='real'
    )
    source_signals[k, :] = seg_k['segment']
    print(f"    Source {k+1}: Frame {k+1}  ({frames[k]['basename']})  "
          f"RMS={np.sqrt(np.mean(seg_k['segment']**2)):.6f}")
print(f"{'='*65}")


In [ ]:
## Section 7b: Run DOA Pipeline & Results


In [ ]:
# ============================================================
# 7b. RUN DOA PIPELINE
# ============================================================
# --- [1/4] Simulate M-element ULA received signals ---
print(f"\n  [1/4] Simulating THz ULA received signals ...")
X_array, A_steer = simulate_ula_signals(
    source_signals, true_doas_deg,
    n_sensors=n_sensors, d_over_lambda=d_over_lambda,
    snr_db=snr_db
)
print(f"         X_array shape  : {X_array.shape}  (M × N)")
print(f"         Steering matrix: {A_steer.shape}  (M × D)")

# --- [2/4] S-Transform + Denoising per element ---
# Compute the frequency band BEFORE denoising so we can pass it in
# for memory-efficient slicing (only keeps ~5 rows instead of N/2)
freq_band = list(range(max(1, peak_freq_bin - freq_band_half),
                       min(N_samples // 2, peak_freq_bin + freq_band_half + 1)))

print(f"\n  [2/4] S-Transform + Denoising per element ...")
S_clean_list, S_opt_list, dn_info = denoise_array_elements(
    X_array,
    gamma_GS=win_gamma_GS, gamma_HY_B=win_gamma_HY_B,
    gamma_HY_F=win_gamma_HY_F, lambda_sq=win_lambda_sq,
   skip_optimization=True, cm_order=cm_order, mu=win_mu,
    keep_freq_bins=freq_band
)
print(f"         TFR shape/element : {S_clean_list[0].shape}")
print(f"         σ (MAD, mean)     : {dn_info['sigma_mean']:.6f}")
print(f"         λ (thresh, mean)  : {dn_info['lam_mean']:.6f}")

# --- [3/4] Build spatial covariance from TFR snapshots ---
# Matrices are already sliced to freq_band — pass same bins for detection
print(f"\n  [3/4] Building spatial covariance ...")
print(f"         Frequency band    : bins {freq_band} (±{freq_band_half} around peak {peak_freq_bin})")
R, n_snapshots = build_spatial_covariance(S_clean_list, freq_bins=freq_band)
print(f"         Covariance R      : {R.shape}")
print(f"         Active snapshots  : {n_snapshots}")

# --- [3b/4] Spatial Smoothing (coherent source decorrelation) ---
# Real THz source waveforms (same BPSK modulation at different distances)
# are inherently correlated.  Spatial smoothing restores full signal-
# subspace rank by averaging overlapping subarray covariances.
subarray_size = n_sensors - n_src + 1   # L = M − D + 1 (optimal)
print(f"\n  [3b/4] Spatial Smoothing (coherent source decorrelation) ...")
R, n_subarrays = spatial_smoothing(R, subarray_size=subarray_size)
eff_resolution = np.degrees(0.886 / (subarray_size * d_over_lambda))
print(f"         Subarray size L   : {subarray_size}  (from M={n_sensors})")
print(f"         Subarrays K       : {n_subarrays}  (≥ D={n_src} for full decorrelation)")
print(f"         Smoothed R        : {R.shape}")
print(f"         Eff. resolution   : {eff_resolution:.1f}°  (post-smoothing)")

# --- [4/4] DOA Estimation ---
print(f"\n  [4/4] Running ROOT-MUSIC, TLS-ESPIRIT, DML, SPICE & MVDR ...")
doa_music, chosen_roots, all_roots, eigenvalues = root_music(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda
)
doa_espirit, _, _ = tls_espirit(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda
)
doa_dml, dml_cost, dml_info = deterministic_ml(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda,
    verbose=True
)
doa_spice, spice_spectrum, spice_info = spice(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda,
    verbose=True
)
doa_mvdr, mvdr_spectrum, mvdr_info = mvdr(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda,
    verbose=True
)

# --- Results ---
M_total = len(eigenvalues)
print(f"\n{'='*140}")
print(f"  DOA   R E S U L T S   (Real THz Data → ULA Simulation)")
print(f"{'='*140}")
print(f"  {'#':<3}  {'True (°)':<10}  {'MUSIC (°)':<11}  {'ESPIRIT (°)':<12}  {'DML (°)':<11}  {'SPICE (°)':<11}  {'MVDR (°)':<11}  {'M-Err':<8}  {'E-Err':<8}  {'D-Err':<8}  {'S-Err':<8}  {'V-Err':<8}")
print(f"  {'-'*3}  {'-'*10}  {'-'*11}  {'-'*12}  {'-'*11}  {'-'*11}  {'-'*11}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
true_sorted = np.sort(true_doas_deg)
for k, (t, m, e, d, s, v) in enumerate(zip(true_sorted, doa_music, doa_espirit, doa_dml, doa_spice, doa_mvdr)):
    print(f"  {k+1:<3}  {t:+.2f}°      {m:+.4f}°    {e:+.4f}°     {d:+.4f}°    {s:+.4f}°    {v:+.4f}°    {abs(t-m):.4f}   {abs(t-e):.4f}   {abs(t-d):.4f}   {abs(t-s):.4f}   {abs(t-v):.4f}")
print(f"-" * 140)
mean_err_m = np.mean(np.abs(true_sorted - doa_music))
mean_err_e = np.mean(np.abs(true_sorted - doa_espirit))
mean_err_d = np.mean(np.abs(true_sorted - doa_dml))
mean_err_s = np.mean(np.abs(true_sorted - doa_spice))
mean_err_v = np.mean(np.abs(true_sorted - doa_mvdr))
print(f"  Mean Error  |  MUSIC: {mean_err_m:.4f}°  |  ESPIRIT: {mean_err_e:.4f}°  |  DML: {mean_err_d:.4f}°  |  SPICE: {mean_err_s:.4f}°  |  MVDR: {mean_err_v:.4f}°")
print(f"  DML cost    |  initial: {dml_info['cost_init']:.6f}   final: {dml_info['cost_final']:.6f}"  
      f"   {'✓ converged' if dml_info['converged'] else '⚠ not converged'}"  
      f"   ({dml_info['n_iterations']} iters)")
print(f"  SPICE       |  σ²={spice_info['sigma2']:.6f}"  
      f"   {'✓ converged' if spice_info['converged'] else '⚠ not converged'}"  
      f"   ({spice_info['n_iterations']} iters)")
print(f"  MVDR        |  cond(R)={mvdr_info['condition_number']:.2e}"  
      f"   dynamic_range={np.max(mvdr_info['P_mvdr_db']) - np.min(mvdr_info['P_mvdr_db']):.1f} dB")
print(f"{'='*75}")

# --- MUSIC Pseudo-Spectrum ---
angles_deg_grid, P_music = music_spectrum(
    R, n_sources=n_src, array_spacing_lambda=d_over_lambda
)
P_db = 10 * np.log10(P_music / np.max(P_music))

print(f"\n  MUSIC Pseudo-Spectrum (Real THz Data):")
print(f"  {'Angle':>8}  {'Power (dB)':>12}  Bar")
print(f"  " + "-" * 50)
step = len(angles_deg_grid) // 36
for i in range(0, len(angles_deg_grid), max(1, step)):
    ang = angles_deg_grid[i]
    pwr = P_db[i]
    bar = "█" * max(0, int((pwr + 40) / 2))
    print(f"  {ang:+7.1f}°  {pwr:+10.2f} dB  {bar}")

# --- Final summary ---
print(f"\n{'='*65}")
print(f"  ✓ Full THz pipeline validated on REAL data")

print(f"    S-Transform → Denoising → DOA")
print(f"  Mean MUSIC DOA error  : {mean_err_m:.4f}°")
print(f"  Mean ESPIRIT DOA error: {mean_err_e:.4f}°")
print(f"  Mean DML DOA error    : {mean_err_d:.4f}°")
print(f"  Mean SPICE DOA error  : {mean_err_s:.4f}°")
print(f"  Mean MVDR DOA error   : {mean_err_v:.4f}°")
print(f"{'='*65}")

"""
# ============================================================
# 8. DOA VISUALIZATION  (links to Section 5)
# ============================================================
print(f"\n\n{'='*65}")
print(f"  Generating DOA Visualizations ...")
print(f"{'='*65}")

viz = importlib.import_module("Visualization")

DATASET_LABEL = "NIST 1.02THz 6cm BPSK"


# --- 8a. Multi-Algorithm SNR sweep → comparison chart ---
print(f"  [8a] Multi-Algorithm SNR vs RMSE Comparison ...")
sweep_results = viz.run_snr_sweep_multi(
    source_signals=source_signals,
    true_doas_deg=true_doas_deg,
    n_sensors=n_sensors,
    d_over_lambda=d_over_lambda,
    freq_band=freq_band,
    gamma_GS=win_gamma_GS,
    gamma_HY_B=win_gamma_HY_B,
    gamma_HY_F=win_gamma_HY_F,
    lambda_sq=win_lambda_sq,
    cm_order=cm_order,
    mu=win_mu,
    snr_range=np.arange(-5, 41, 5),
    n_trials=10,
)

# Multi-algorithm comparison line chart (paper-ready)
viz.plot_multi_algo_rmse(
    sweep_results,
    dataset_label=DATASET_LABEL,
    n_sensors=n_sensors,
    d_over_lambda=d_over_lambda,
    n_trials=10,
)

# Per-source RMSE ridgeline (ROOT-MUSIC baseline)
viz.plot_per_source_rmse(
    snr_db=sweep_results['snr_db'],
    rmse_per_source=sweep_results['per_algo']['ROOT-MUSIC']['rmse_per_source'],
    true_doas_deg=true_doas_deg,
    dataset_label=DATASET_LABEL,
    n_sensors=n_sensors,
    d_over_lambda=d_over_lambda,
    n_trials=10,
)

# 3D accuracy surface (ROOT-MUSIC baseline)
viz.plot_rmse_3d_surface(
    snr_db=sweep_results['snr_db'],
    rmse_per_source=sweep_results['per_algo']['ROOT-MUSIC']['rmse_per_source'],
    true_doas_deg=true_doas_deg,
    dataset_label=DATASET_LABEL,
)
"""
print(f"\n{'='*65}")
print(f"  ✓ All DOA visualizations rendered")
print(f"{'='*65}")
